In [1]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================

# ============================================================
# CORE NUMERICAL & DATA HANDLING
# ============================================================
import numpy as np
import pandas as pd

# ============================================================
# VISUALIZATION
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# STATISTICAL TESTING
# ============================================================
from scipy import stats

# ============================================================
# SKLEARN: MODEL SELECTION & VALIDATION
# ============================================================
from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split
)

# ============================================================
# SKLEARN: MODELS
# ============================================================
from sklearn.ensemble import RandomForestClassifier

# ============================================================
# SKLEARN: METRICS & FEATURE IMPORTANCE
# ============================================================
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    mutual_info_score
)
from sklearn.inspection import permutation_importance

# ============================================================
# GRADIENT BOOSTING
# ============================================================
import xgboost as xgb

# ============================================================
# RULE / ILP / STRUCTURAL UTILITIES
# ============================================================
import itertools
from collections import defaultdict
from dataclasses import dataclass
from typing import List, Dict, Tuple, Set

# ============================================================
# WARNINGS CONTROL
# ============================================================
import warnings
warnings.filterwarnings("ignore")


In [2]:
# ============================================================
# CELL 2: LOAD & INSPECT DATA
# ============================================================

# Load dataset
path = "/content/sample_data/bank-full.csv"
df = pd.read_csv(path, sep=";")

print(f"Shape: {df.shape}")
df.head()


Shape: (45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [3]:
# ============================================================
# CELL 4: EXAMINE CATAGORICAL COLUMNS
# ============================================================
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("📌 CATEGORICAL COLUMNS\n" + "-"*60)
categorical_cols

for col in categorical_cols:
    print(f"\n🔎 Column: {col}")
    print("-" * 40)

    values = (
        df[col]
        .astype(str)
        .value_counts(dropna=False)
        .sort_index()
    )

    print(values)


📌 CATEGORICAL COLUMNS
------------------------------------------------------------

🔎 Column: job
----------------------------------------
job
admin.           5171
blue-collar      9732
entrepreneur     1487
housemaid        1240
management       9458
retired          2264
self-employed    1579
services         4154
student           938
technician       7597
unemployed       1303
unknown           288
Name: count, dtype: int64

🔎 Column: marital
----------------------------------------
marital
divorced     5207
married     27214
single      12790
Name: count, dtype: int64

🔎 Column: education
----------------------------------------
education
primary       6851
secondary    23202
tertiary     13301
unknown       1857
Name: count, dtype: int64

🔎 Column: default
----------------------------------------
default
no     44396
yes      815
Name: count, dtype: int64

🔎 Column: housing
----------------------------------------
housing
no     20081
yes    25130
Name: count, dtype: int64

🔎 Co

In [4]:
# ============================================================
# CELL 5: PREPROCESSING + FEATURE ENGINEERING + TUNING
# ============================================================

print("\n" + "="*80)
print("🔧 CLEAN PREPROCESSING FROM SCRATCH (NUMERIC-ONLY)")
print("="*80)

print(f"\nOriginal data shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# --------------------------------------------------
# 1. CREATE CLEAN df_proc FRAME
# --------------------------------------------------
print("\n" + "="*40)
print("1. CREATING CLEAN df_proc")
print("="*40)

df_proc = df.copy()

print(f"\nInitial shape: {df_proc.shape}")
print(f"Columns: {list(df_proc.columns)}")

# --------------------------------------------------
# 1.1 Target encoding (STRICT)
# --------------------------------------------------
print(f"\n🔍 Target column 'y' before conversion:")
print(df_proc['y'].value_counts())

df_proc['y'] = df_proc['y'].map({'yes': 1, 'no': 0})
assert df_proc['y'].notna().all(), "❌ Unexpected values in target y"
df_proc['y'] = df_proc['y'].astype('int8')

print(f"\n✅ Target 'y' converted:")
print(df_proc['y'].value_counts())

# --------------------------------------------------
# 1.2 Binary categorical encoding
# --------------------------------------------------
binary_cols = ['default', 'housing', 'loan']
binary_map = {'no': 0, 'yes': 1}

print(f"\n🔍 Binary columns before conversion:")
for col in binary_cols:
    print(f"{col}:\n{df_proc[col].value_counts()}")

for col in binary_cols:
    df_proc[col] = df_proc[col].map(binary_map)
    assert df_proc[col].notna().all(), f"❌ Unexpected values in {col}"
    df_proc[col] = df_proc[col].astype('int8')

print(f"\n✅ Binary columns converted:")
for col in binary_cols:
    print(f"{col}: {df_proc[col].unique()}")

# --------------------------------------------------
# 1.3 Month → ordinal
# --------------------------------------------------
print(f"\n🔍 Month column before conversion:")
print(df_proc['month'].value_counts().sort_index())

month_map = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4,
    'may': 5, 'jun': 6, 'jul': 7, 'aug': 8,
    'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}

df_proc['month'] = df_proc['month'].map(month_map)
assert df_proc['month'].notna().all(), "❌ Unexpected month values"

df_proc['month'] = df_proc['month'].astype('int8')

print(f"\n✅ Month converted to ordinal")

# --------------------------------------------------
# 1.4 POUTCOME - EXPLICIT MAPPING
# --------------------------------------------------
print(f"\n🔍 poutcome column before conversion:")
print(df_proc['poutcome'].value_counts())

poutcome_map = {
    'unknown': 0,
    'failure': 1,
    'other': 2,
    'success': 3
}

df_proc['poutcome'] = df_proc['poutcome'].map(poutcome_map)
assert df_proc['poutcome'].notna().all(), "❌ Unexpected poutcome values"
df_proc['poutcome'] = df_proc['poutcome'].astype('int8')

print(f"\n✅ poutcome converted:")
print(df_proc['poutcome'].value_counts())

# --------------------------------------------------
# 1.5 CONTACT - ORDINAL MAPPING
# --------------------------------------------------
print(f"\n🔍 contact column before conversion:")
print(df_proc['contact'].value_counts())

contact_map = {
    'cellular': 0,
    'telephone': 1,
    'unknown': 2
}

df_proc['contact'] = df_proc['contact'].map(contact_map)
assert df_proc['contact'].notna().all(), "❌ Unexpected contact values"
df_proc['contact'] = df_proc['contact'].astype('int8')

print(f"\n✅ contact converted to ordinal:")
print(df_proc['contact'].value_counts().sort_index())

# --------------------------------------------------
# 1.6 Ordinal / Label encoding REMAINING categoricals
# --------------------------------------------------
categorical_cols = df_proc.select_dtypes(include=['object']).columns.tolist()

print(f"\n🔍 Ordinal-encoding remaining categoricals:")
print(categorical_cols)

for col in categorical_cols:
    df_proc[col] = df_proc[col].astype('category').cat.codes
    df_proc[col] = df_proc[col].astype('int16')

# --------------------------------------------------
# 1.7 HARD TYPE GUARANTEE
# --------------------------------------------------
non_numeric = df_proc.select_dtypes(exclude=['number']).columns.tolist()
assert len(non_numeric) == 0, f"❌ NON-NUMERIC COLUMNS REMAIN: {non_numeric}"

print("\n✅ ALL COLUMNS ARE NUMERIC")

# --------------------------------------------------
# 1.8 Sanity checks
# --------------------------------------------------
print(f"\n📊 Data types in df_proc:")
for dtype in df_proc.dtypes.unique():
    cols = df_proc.columns[df_proc.dtypes == dtype].tolist()
    print(f"{dtype}: {cols}")

print(f"\n🔍 Checking for null values:")
assert df_proc.isnull().sum().sum() == 0, "❌ Nulls detected"
print("✅ No null values")

print(f"\n🔍 Checking for duplicates:")
dup_count = df_proc.duplicated().sum()
print(f"Exact duplicate rows: {dup_count} ({dup_count/len(df_proc)*100:.2f}%)")

print(f"\n✅ df_proc created successfully!")
print(f"Shape: {df_proc.shape}")
print(f"Memory usage: {df_proc.memory_usage().sum() / 1024**2:.2f} MB")




🔧 CLEAN PREPROCESSING FROM SCRATCH (NUMERIC-ONLY)

Original data shape: (45211, 17)
Columns: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

1. CREATING CLEAN df_proc

Initial shape: (45211, 17)
Columns: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']

🔍 Target column 'y' before conversion:
y
no     39922
yes     5289
Name: count, dtype: int64

✅ Target 'y' converted:
y
0    39922
1     5289
Name: count, dtype: int64

🔍 Binary columns before conversion:
default:
default
no     44396
yes      815
Name: count, dtype: int64
housing:
housing
yes    25130
no     20081
Name: count, dtype: int64
loan:
loan
no     37967
yes     7244
Name: count, dtype: int64

✅ Binary columns converted:
default: [0 1]
housing: [1 0]
loan: [0 1]

🔍 Month column before con

In [6]:
# ============================================================
# CELL 6: GLOBAL TRAIN / TEST SPLIT (AUTHORITATIVE)
# PURPOSE:
# - Single, fixed split shared by ALL models
# - Consistent evaluation & meta-alignment
# ============================================================

print("\n" + "="*80)
print("📐 GLOBAL TRAIN / TEST SPLIT (SHARED ACROSS ALL MODELS)")
print("="*80)

from sklearn.model_selection import train_test_split, StratifiedKFold

# --------------------------------------------------
# 6.1 Define features and target
# --------------------------------------------------
X_all = df_proc.drop(columns=['y'])
y_all = df_proc['y']

print(f"\nTotal samples: {len(df_proc)}")
print(f"Positive rate: {y_all.mean():.4f}")
print(f"Class counts:\n{y_all.value_counts()}")

# --------------------------------------------------
# 6.2 Global holdout split (DO THIS ONCE)
# --------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_all,
    y_all,
    test_size=0.20,
    stratify=y_all,
    random_state=42
)

# Save indices explicitly (IMPORTANT for meta alignment)
train_idx = X_train.index.values
test_idx  = X_test.index.values

print("\n✅ Global split created")
print(f"Train shape: {X_train.shape}")
print(f"Test  shape: {X_test.shape}")

print(f"\nTrain positives: {y_train.sum()} ({y_train.mean():.4f})")
print(f"Test  positives: {y_test.sum()} ({y_test.mean():.4f})")

# --------------------------------------------------
# 6.3 Shared 10-fold CV object (train-only)
# --------------------------------------------------
cv_10 = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

print("\n✅ Shared 10-fold Stratified CV configured")
print("• CV is applied ONLY on training data")
print("• Test set remains untouched until final evaluation")

# --------------------------------------------------
# 6.4 Sanity check: no overlap
# --------------------------------------------------
assert len(set(train_idx) & set(test_idx)) == 0, "❌ Train/Test overlap detected!"

print("\n🔒 Sanity check passed: no train/test leakage")

# --------------------------------------------------
# 6.5 Canonical objects to reuse everywhere
# --------------------------------------------------
GLOBAL_SPLIT = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'train_idx': train_idx,
    'test_idx': test_idx,
    'cv_10': cv_10,
    'random_state': 42
}

print("\n📦 GLOBAL_SPLIT object created")
print("Use this for LR / BRW / EBM / Meta models")

print("\n" + "="*80)
print("✅ GLOBAL SPLIT READY — ALL MODELS MUST USE THIS")
print("="*80)



📐 GLOBAL TRAIN / TEST SPLIT (SHARED ACROSS ALL MODELS)

Total samples: 45211
Positive rate: 0.1170
Class counts:
y
0    39922
1     5289
Name: count, dtype: int64

✅ Global split created
Train shape: (36168, 16)
Test  shape: (9043, 16)

Train positives: 4231 (0.1170)
Test  positives: 1058 (0.1170)

✅ Shared 10-fold Stratified CV configured
• CV is applied ONLY on training data
• Test set remains untouched until final evaluation

🔒 Sanity check passed: no train/test leakage

📦 GLOBAL_SPLIT object created
Use this for LR / BRW / EBM / Meta models

✅ GLOBAL SPLIT READY — ALL MODELS MUST USE THIS


In [7]:
# ============================================================
# CELL 9: MODEL PERSISTENCE & PREPARATION FOR ENSEMBLE
# (USES GLOBAL_SPLIT FROM CELL 6)
# ============================================================

print("=" * 80)
print("💾 CELL 9: MODEL PERSISTENCE & PREPARATION FOR ENSEMBLE")
print("=" * 80)

import pickle
import joblib
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, roc_curve, auc,
    precision_score, recall_score,
    f1_score, roc_auc_score
)

# --------------------------------------------------
# 0. LOAD GLOBAL SPLIT (AUTHORITATIVE)
# --------------------------------------------------
print("\n🔒 USING GLOBAL_SPLIT FROM CELL 6")
print("-" * 50)

X_train = GLOBAL_SPLIT['X_train']
X_test  = GLOBAL_SPLIT['X_test']
y_train = GLOBAL_SPLIT['y_train']
y_test  = GLOBAL_SPLIT['y_test']

train_idx = GLOBAL_SPLIT['train_idx']
test_idx  = GLOBAL_SPLIT['test_idx']

print(f"Train shape: {X_train.shape}")
print(f"Test  shape: {X_test.shape}")
print(f"Train positives: {y_train.sum()} ({y_train.mean():.3f})")
print(f"Test  positives: {y_test.sum()} ({y_test.mean():.3f})")

# --------------------------------------------------
# 1. SCALE FEATURES (TRAIN ONLY)
# --------------------------------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("\n✅ Features scaled using StandardScaler (train-fit only)")

# --------------------------------------------------
# 2. TRAIN LOGISTIC REGRESSION (RECALL-ORIENTED BASELINE)
# --------------------------------------------------
print("\n🤖 TRAINING LOGISTIC REGRESSION MODEL")
print("-" * 50)

lr_model = LogisticRegression(
    C=1.0,
    penalty='l2',
    solver='liblinear',
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred = lr_model.predict(X_test_scaled)
y_pred_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

metrics_tuned = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1-Score': f1_score(y_test, y_pred),
    'ROC-AUC': roc_auc_score(y_test, y_pred_proba)
}

print("\n✅ Test-set performance (GLOBAL_SPLIT):")
for k, v in metrics_tuned.items():
    print(f"  {k}: {v:.4f}")

# --------------------------------------------------
# 3. FEATURE IMPORTANCE
# --------------------------------------------------
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_model.coef_[0],
    'Abs_Coefficient': np.abs(lr_model.coef_[0])
}).sort_values('Abs_Coefficient', ascending=False)

print("\nTop 10 LR features:")
print(feature_importance.head(10).to_string(index=False))

# --------------------------------------------------
# 4. ENSEMBLE-READY OUTPUTS (ALIGNED BY INDEX)
# --------------------------------------------------
lr_X_train = lr_model.predict_proba(X_train_scaled)[:, 1]
lr_X_test  = lr_model.predict_proba(X_test_scaled)[:, 1]

print("\n📊 LR prediction stats:")
print(f"Train range: [{lr_X_train.min():.4f}, {lr_X_train.max():.4f}]")
print(f"Test  range: [{lr_X_test.min():.4f}, {lr_X_test.max():.4f}]")

# --------------------------------------------------
# 5. CREATE ENSEMBLE DICTIONARY (META-SAFE)
# --------------------------------------------------
lr_ensemble_dict = {
    'model': lr_model,
    'model_name': 'logistic_regression',
    'scaler': scaler,
    'feature_names': X_train.columns.tolist(),
    'train_idx': train_idx,
    'test_idx': test_idx,
    'train_predictions': lr_X_train,
    'test_predictions': lr_X_test,
    'train_labels': y_train.values,
    'test_labels': y_test.values,
    'performance_metrics': metrics_tuned,
    'training_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

print("\n✅ Ensemble dictionary aligned with GLOBAL_SPLIT")

# --------------------------------------------------
# 6. SAVE MODEL & ENSEMBLE OBJECTS
# --------------------------------------------------
base_path = "./models"
os.makedirs(base_path, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_path = f"{base_path}/lr_model_{timestamp}.pkl"
ensemble_path = f"{base_path}/lr_ensemble_{timestamp}.joblib"

with open(model_path, 'wb') as f:
    pickle.dump(lr_model, f)

joblib.dump(lr_ensemble_dict, ensemble_path)

print(f"\n💾 Saved:")
print(f"• Model:    {model_path}")
print(f"• Ensemble: {ensemble_path}")

print("\n🎯 LR is now GLOBAL-SPLIT SAFE and META-READY")


💾 CELL 9: MODEL PERSISTENCE & PREPARATION FOR ENSEMBLE

🔒 USING GLOBAL_SPLIT FROM CELL 6
--------------------------------------------------
Train shape: (36168, 16)
Test  shape: (9043, 16)
Train positives: 4231 (0.117)
Test  positives: 1058 (0.117)

✅ Features scaled using StandardScaler (train-fit only)

🤖 TRAINING LOGISTIC REGRESSION MODEL
--------------------------------------------------

✅ Test-set performance (GLOBAL_SPLIT):
  Accuracy: 0.8236
  Precision: 0.3792
  Recall: 0.7968
  F1-Score: 0.5139
  ROC-AUC: 0.8886

Top 10 LR features:
  Feature  Coefficient  Abs_Coefficient
 duration     1.413079         1.413079
 poutcome     0.593572         0.593572
  contact    -0.541589         0.541589
  housing    -0.508216         0.508216
 campaign    -0.368983         0.368983
     loan    -0.265189         0.265189
education     0.155596         0.155596
    pdays    -0.137874         0.137874
  marital     0.134966         0.134966
  balance     0.088693         0.088693

📊 LR predi

In [125]:
# ============================================================
# CELL 7: GLASS-BRW STATE-BASED FEATURE ENGINEERING
# ============================================================

def engineer_features_bank(df_proc: pd.DataFrame) -> pd.DataFrame:
    """
    RF-aligned feature engineering for GLASS-BRW.

    INPUT  : df_proc (FULLY NUMERIC preprocessed frame)
    OUTPUT : df_eng  (Rule-safe binary states + target y)
    """

    print("🔍 Engineering RF-aligned features...")

    # =========================================================
    # DURATION REGIMES (RF-FAITHFUL)
    # =========================================================
    df_proc["dur_low"]  = (df_proc["duration"] <= 200).astype("int8")
    df_proc["dur_mid"]  = ((df_proc["duration"] > 200) & (df_proc["duration"] <= 300)).astype("int8")
    df_proc["dur_high"] = (df_proc["duration"] > 300).astype("int8")

    # =========================================================
    # PDAYS MEMORY STATES (COARSE, STABLE)
    # =========================================================
    df_proc["pdays_never"] = (df_proc["pdays"] <= 0).astype("int8")
    df_proc["pdays_1_5"]   = ((df_proc["pdays"] > 0) & (df_proc["pdays"] <= 5)).astype("int8")

    # =========================================================
    # MONTH REGIMES (MUTUALLY EXCLUSIVE)
    # =========================================================
    df_proc["month_early"]      = df_proc["month"].isin([1, 2, 3, 4]).astype("int8")
    df_proc["month_late_spring"] = df_proc["month"].isin([5, 6]).astype("int8")
    df_proc["month_summer"]     = df_proc["month"].isin([7, 8]).astype("int8")
    df_proc["month_fall"]       = df_proc["month"].isin([9, 10]).astype("int8")
    df_proc["month_year_end"]   = df_proc["month"].isin([11, 12]).astype("int8")

    # =========================================================
    # SEMANTIC / BEHAVIORAL STATES
    # =========================================================
    df_proc["previous_ge1"] = (df_proc["previous"] >= 1).astype("int8")

    df_proc["engagement_score"] = (
        df_proc["previous_ge1"]
        + df_proc["pdays_1_5"]
        + df_proc["dur_high"]
        + (df_proc["poutcome"] == 3).astype("int8")
    ).astype("int8")

    df_proc["fatigue_score"] = (
        (df_proc["campaign"] >= 5).astype("int8")
        + (df_proc["campaign"] >= 8).astype("int8")
        + df_proc["dur_low"]
        + df_proc["pdays_never"]
    ).astype("int8")

    print("✅ RF-driven semantic states created")

    # =========================================================
    # BUILD df_eng (FINAL RULE INPUT)
    # =========================================================
    df_eng = pd.DataFrame(index=df_proc.index)

    # --- Target
    df_eng["y"] = df_proc["y"].astype("int8")

    # --- Duration
    df_eng["dur_low"]  = df_proc["dur_low"]
    df_eng["dur_mid"]  = df_proc["dur_mid"]
    df_eng["dur_high"] = df_proc["dur_high"]


    # --- Temporal opportunity
    df_eng["month_summer"]   = df_proc["month_summer"]
    df_eng["month_fall"]     = df_proc["month_fall"]
    df_eng["month_year_end"] = df_proc["month_year_end"]
    df_eng["month_early"] = df_proc["month_early"]
    df_eng["month_late_spring"] = df_proc["month_late_spring"]

    # --- Behavioral semantics
    df_eng["engagement_low"] = (df_proc["engagement_score"] <= 1).astype("int8")
    df_eng["fatigue_low"]    = (df_proc["fatigue_score"] <= 1).astype("int8")

    # --- Context + memory
    df_eng["housing"]          = df_proc["housing"].astype("int8")
    df_eng["poutcome_success"] = (df_proc["poutcome"] == 3).astype("int8")
    df_eng["pdays_never"]      = df_proc["pdays_never"]
    #df_eng["pdays_1_5"]        = df_proc["pdays_1_5"]

    print(f"✅ Engineered {df_eng.shape[1] - 1} RF-aligned features (+ y)")
    return df_eng


In [126]:
# ============================================================
# CELL 9: RULE DATA STRUCTURE
# ============================================================

@dataclass
class Rule:
    """
    Atomic rule representation for GLASS-BRW.

    A rule consists of:
    - segment: conjunction of (feature, value) conditions
    - predicted_class: 0 (NOT_SUBSCRIBE) or 1 (SUBSCRIBE)
    - complexity: number of conditions (k ≤ 3)
    """
    rule_id: int
    segment: Set[Tuple[str, str]]
    predicted_class: int  # 0 = NOT_SUBSCRIBE, 1 = SUBSCRIBE
    complexity: int

    # Metrics computed during validation
    precision: float = 0.0
    recall: float = 0.0
    coverage: float = 0.0
    stability: float = 0.0
    interpretability: float = 0.0
    boundary_ambiguity: float = 0.0
    rf_alignment: float = 0.0  # CHANGED: ebm_overlap -> rf_alignment

    def __hash__(self):
        return hash((self.rule_id, frozenset(self.segment), self.predicted_class))

    def __eq__(self, other):
        return self.rule_id == other.rule_id



In [127]:
# ============================================================
# CELL 10: GLASS-BRW: RF-BASED RULE EXTRACTION - PART 1
# Core Data Structures and Segment Builder
# ============================================================

from dataclasses import dataclass
from typing import Set, Tuple, List, Dict
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# ============================================================
# SEGMENT BUILDER - RF-ALIGNED FEATURES
# ============================================================

class BankSegmentBuilder:
    """
    RF-aligned binary segment builder for GLASS-BRW.
    Single-tier version (rule-safe).

    Uses only binary, interpretable state features.
    """

    SEGMENT_FEATURES = [
        # Duration intent
        "dur_low",
        "dur_mid",
        "dur_high",

        #   Temporal opportunity
        "month_early",
        "month_late_spring",
        "month_summer",
        "month_fall",
        "month_year_end",

        # Behavioral semantics
        "engagement_low",
        "fatigue_low",
        "poutcome_success",

        # Context
        "housing",

        # Memory
        "pdays_never",
       #"pdays_1_5"

    ]


    def assign_segments(self, X: pd.DataFrame) -> pd.DataFrame:
        """
        Extract binary segment features from input data.

        Args:
            X: DataFrame with engineered binary features

        Returns:
            Binary DataFrame used for rule generation
        """
        X = X.drop(columns=["y", "y_bin"], errors="ignore")

        # Filter to available columns
        cols = [c for c in self.SEGMENT_FEATURES if c in X.columns]

        if len(cols) == 0:
            raise ValueError("No valid segment features found in input")

        segments = X[cols].copy()

        # Enforce binary contract
        for col in segments.columns:
            vals = set(segments[col].dropna().unique())
            if not vals.issubset({0, 1}):
                raise ValueError(f"Non-binary segment feature: {col} -> {vals}")

        return segments.astype("int8")

In [128]:
# ============================================================
# CELL 11: GLASS-BRW: RF-BASED RULE EXTRACTION - PART 2
# Rule Generator with RF-Guided Feature Selection
# ============================================================

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

class RuleGenerator:
    """
    Generates candidate rules using RF-guided feature importance
    and beam search over segment lattice.

    Key changes from EBM version:
    - Uses RF feature importance for initial feature ranking
    - Computes RF-alignment score for validation
    - Maintains class-conditional constraints from GLASS-BRW spec
    """

    def __init__(
        self,
        min_support=50,

        # GLOBAL bounds
        max_complexity=3,
        min_complexity=3,

        # CLASS-CONDITIONAL complexity
        min_complexity_subscribe=3,
        min_complexity_not_subscribe=3,

        # CLASS-CONDITIONAL coverage caps
        max_coverage_subscribe=0.05,
        max_coverage_not_subscribe=0.15,

        # CLASS-CONDITIONAL precision floors
        min_precision_subscribe=0.75,
        min_precision_not_subscribe=0.75,

        # CLASS-CONDITIONAL PRECISION CAPS
        max_precision_subscribe=0.95,
        max_precision_not_subscribe=0.80,

        # CLASS-CONDITIONAL recall floors
        min_recall_subscribe=0.25,
        min_recall_not_subscribe=0.32,

        # CLASS-CONDITIONAL AUC floors (rule-local AUC)
        min_auc_subscribe=0.50,
        min_auc_not_subscribe=0.50,

        # Beam search parameters
        mode="strict",
        beam_width=100,
        min_precision_gain=0.0,
        min_seed_precision_subscribe=0.60,
        min_seed_precision_not_subscribe=0.60,

        # Feature tier identification
        tier1_prefixes=("dur", "month", "pdays",
                        "fatigue", "engagement", "poutcome"),

        # RF-specific parameters
        rf_model=None,  # Pre-trained RF model
        feature_importance_threshold=0.01,  # Min importance for feature consideration
    ):
        # Support and complexity
        self.min_support = min_support
        self.max_complexity = max_complexity
        self.min_complexity = min_complexity

        # Class-conditional complexity
        self.min_complexity_subscribe = min_complexity_subscribe
        self.min_complexity_not_subscribe = min_complexity_not_subscribe

        # Class-conditional coverage
        self.max_coverage_subscribe = max_coverage_subscribe
        self.max_coverage_not_subscribe = max_coverage_not_subscribe

        # Class-conditional precision
        self.min_precision_subscribe = min_precision_subscribe
        self.min_precision_not_subscribe = min_precision_not_subscribe

        # Class-conditional minimum recall
        self.min_recall_subscribe = min_recall_subscribe
        self.min_recall_not_subscribe = min_recall_not_subscribe

        # Class-conditional max recall
        self.max_precision_subscribe = max_precision_subscribe
        self.max_precision_not_subscribe = max_precision_not_subscribe

        # Class-conditional AUC
        self.min_auc_subscribe = min_auc_subscribe
        self.min_auc_not_subscribe = min_auc_not_subscribe

        # Beam search
        self.mode = mode
        self.beam_width = beam_width
        self.min_precision_gain = min_precision_gain
        self.min_seed_precision_subscribe = min_seed_precision_subscribe
        self.min_seed_precision_not_subscribe = min_seed_precision_not_subscribe

        # Feature semantics
        self.tier1_prefixes = tier1_prefixes

        # RF-specific
        self.rf_model = rf_model
        self.feature_importance_threshold = feature_importance_threshold
        self.feature_importances_ = None

    def _compute_rf_feature_importance(self, segments_df, y):
        """
        Extract feature importance from RF model or train new one.

        Args:
            segments_df: Binary segment features
            y: Target labels

        Returns:
            Dict mapping feature names to importance scores
        """
        if self.rf_model is None:
            # Train RF for importance extraction
            print("Training RF for feature importance...")
            self.rf_model = RandomForestClassifier(
                n_estimators=100,
                max_depth=8,
                min_samples_split=50,
                min_samples_leaf=25,
                random_state=42,
                n_jobs=-1
            )
            self.rf_model.fit(segments_df, y)

        # Extract importance scores
        importances = dict(zip(
            segments_df.columns,
            self.rf_model.feature_importances_
        ))

        self.feature_importances_ = importances

        # Log top features
        top_features = sorted(importances.items(), key=lambda x: x[1], reverse=True)[:10]
        print("\nTop 10 RF features by importance:")
        for feat, imp in top_features:
            print(f"  {feat}: {imp:.4f}")

        return importances

    def _compute_rule_mask(self, segment, segments_df):
        """Create boolean mask for samples matching rule segment."""
        mask = pd.Series(True, index=segments_df.index)
        for f, l in segment:
            mask &= (segments_df[f] == l)
        return mask

    def _compute_precision(self, mask, y, cls):
        """Compute precision for rule on given class."""
        if mask.sum() == 0:
            return 0.0
        return (y[mask] == cls).mean()

    def _compute_recall(self, mask, y, cls):
        """Compute recall for rule on given class."""
        total_class = (y == cls).sum()
        if total_class == 0:
            return 0.0
        true_positives = ((y == cls) & mask).sum()
        return true_positives / total_class

    def _compute_rule_auc(self, mask, y, cls):
        """
        Compute local AUC for rule predictions.

        Treats rule as binary classifier: 1 if matched, 0 otherwise.
        """
        if mask.sum() == 0 or mask.sum() == len(mask):
            return 0.5

        try:
            y_true = (y == cls).astype(int)
            y_score = mask.astype(int)
            return roc_auc_score(y_true, y_score)
        except:
            return 0.5

    def _has_tier1_feature(self, segment):
        """Check if segment contains Tier 1 behavioral feature."""
        feats = {f for f, _ in segment}
        return any(any(f.startswith(p) for p in self.tier1_prefixes) for f in feats)

    def _is_valid_semantic(self, segment):
        """Validate semantic composition of segment."""
        return self._has_tier1_feature(segment)

    def _score(self, p, r, c):
        """
        Compute beam search priority score.

        Emphasizes precision² × recall × coverage
        to favor high-precision, high-impact rules.
        """
        return (p ** 2) * r * c

    def _get_feature_order(self, segments_df, y):
        """
        Determine feature expansion order using RF importance.

        Returns features sorted by importance (descending).
        Features below threshold are excluded.
        """
        importances = self._compute_rf_feature_importance(segments_df, y)

        # Filter by threshold
        important_features = [
            f for f, imp in importances.items()
            if imp >= self.feature_importance_threshold
        ]

        # Sort by importance
        important_features.sort(key=lambda f: importances[f], reverse=True)

        print(f"\nUsing {len(important_features)}/{len(segments_df.columns)} features above threshold")

        return important_features

    def generate_candidates(self, segments_df, y):
        """
        Generate candidate rules via RF-guided beam search.

        Process:
        1. Compute RF feature importance
        2. Seed rules from top features
        3. Beam search expansion with precision/recall gates
        4. Return rules meeting complexity requirements

        Args:
            segments_df: Binary segment features (N × F)
            y: Target labels (N,)

        Returns:
            List of Rule objects passing all filters
        """
        # Get RF-ranked features
        features = self._get_feature_order(segments_df, y)
        N = len(segments_df)

        print(f"\nBeam search configuration:")
        print(f"  Beam width: {self.beam_width}")
        print(f"  Complexity: {self.min_complexity}–{self.max_complexity}")
        print(f"  Min support: {self.min_support} samples")

        rule_id = 0
        all_candidates = []
        current = {0: [], 1: []}

        # ============================================================
        # DEPTH 1: SEED RULES FROM TOP FEATURES
        # ============================================================
        print("\n" + "="*60)
        print("DEPTH 1: Seeding rules from RF-important features")
        print("="*60)

        for f in features:
            for l in (1, 0):  # Binary: presence (1) or absence (0)
                seg = {(f, l)}

                # Semantic validation
                if not self._is_valid_semantic(seg):
                    continue

                mask = self._compute_rule_mask(seg, segments_df)
                supp = mask.sum()
                if supp < self.min_support:
                    continue

                cov = supp / N

                # Generate rules for both classes
                for cls in (0, 1):
                    p = self._compute_precision(mask, y, cls)
                    r = self._compute_recall(mask, y, cls)

                    # Class-conditional seed filters
                    if cls == 1 and p < self.min_seed_precision_subscribe:
                        continue
                    if cls == 0 and p < self.min_seed_precision_not_subscribe:
                        continue
                    if cls == 1 and r < self.min_recall_subscribe:
                        continue
                    if cls == 0 and r < self.min_recall_not_subscribe:
                        continue

                    rule = Rule(
                        rule_id=rule_id,
                        segment=seg,
                        predicted_class=cls,
                        complexity=1,
                        interpretability=1.0
                    )

                    # Store metrics for beam scoring
                    rule._p, rule._r, rule._c = p, r, cov
                    rule._s = self._score(p, r, cov)

                    current[cls].append(rule)
                    rule_id += 1

        # Beam pruning
        for cls in (0, 1):
            current[cls] = sorted(current[cls], key=lambda r: r._s, reverse=True)[:self.beam_width]

        print(f"\nSeeds after beam prune:")
        print(f"  SUBSCRIBE: {len(current[1])} rules")
        print(f"  NOT_SUBSCRIBE: {len(current[0])} rules")

        # ============================================================
        # DEPTH 2+: BEAM SEARCH EXPANSION
        # ============================================================
        for depth in range(2, self.max_complexity + 1):
            print(f"\n{'='*60}")
            print(f"DEPTH {depth}: Expanding rules")
            print("="*60)

            next_rules = {0: [], 1: []}
            expansions = 0

            for cls in (0, 1):
                for parent in current[cls]:
                    used = {f for f, _ in parent.segment}

                    # Try adding each unused feature
                    for f in features:
                        if f in used:
                            continue

                        for l in (1, 0):
                            seg = parent.segment | {(f, l)}

                            # Semantic validation
                            if not self._is_valid_semantic(seg):
                                continue

                            mask = self._compute_rule_mask(seg, segments_df)
                            supp = mask.sum()
                            if supp < self.min_support:
                                continue

                            p = self._compute_precision(mask, y, cls)
                            r = self._compute_recall(mask, y, cls)

                            # Precision monotonicity check
                            if p < parent._p - self.min_precision_gain:
                                continue

                            # Class-conditional recall gates
                            if cls == 1 and r < self.min_recall_subscribe:
                                continue
                            if cls == 0 and r < self.min_recall_not_subscribe:
                                continue

                            cov = supp / N
                            rule = Rule(
                                rule_id=rule_id,
                                segment=seg,
                                predicted_class=cls,
                                complexity=depth,
                                interpretability=1.0 / depth
                            )

                            rule._p, rule._r, rule._c = p, r, cov
                            rule._s = self._score(p, r, cov)

                            next_rules[cls].append(rule)
                            rule_id += 1
                            expansions += 1

            if expansions == 0:
                print(f"No valid expansions — stopping early at depth {depth-1}")
                break

            # Beam pruning
            for cls in (0, 1):
                next_rules[cls] = sorted(next_rules[cls], key=lambda r: r._s, reverse=True)[:self.beam_width]

                # Add to candidates if complexity requirement met
                if depth >= self.min_complexity:
                    all_candidates.extend(next_rules[cls])

            print(f"\nAfter beam prune:")
            print(f"  SUBSCRIBE: {len(next_rules[1])} rules")
            print(f"  NOT_SUBSCRIBE: {len(next_rules[0])} rules")

            current = next_rules

        print(f"\n{'='*60}")
        print(f"✅ Generated {len(all_candidates)} candidate rules (k ≥ {self.min_complexity})")
        print("="*60)

        return all_candidates


In [138]:
# ============================================================
# CELL 12: GLASS-BRW: RF-BASED RULE EXTRACTION - PART 3
# Rule Evaluation and Validation Metrics
# ============================================================

import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score
from collections import defaultdict

class RuleEvaluator:
    """
    Compute validation-based quality metrics for candidate rules.

    Key changes from EBM version:
    - Replaced ebm_overlap with rf_alignment
    - Replaced boundary_ambiguity with rf_confidence_score
    - Uses RF predictions for complementarity metrics
    """

    def __init__(self, segment_builder, min_support=30):
        self.segment_builder = segment_builder
        self.min_support = min_support

    def _deduplicate_rules(self, rules):
        """
        Remove semantically identical rules.

        Identity = (predicted_class, segment conditions).
        Keeps the highest-precision version of each rule.
        """
        unique = {}

        for r in rules:
            key = (r.predicted_class, frozenset(r.segment))

            if key not in unique:
                unique[key] = r
            else:
                # Keep the better rule (prefer higher precision, then coverage)
                if (
                    r.precision > unique[key].precision or
                    (
                        r.precision == unique[key].precision and
                        r.coverage > unique[key].coverage
                    )
                ):
                    unique[key] = r

        return list(unique.values())


    def match_rule(self, rule, segments_df):
        """
        Create boolean mask for samples matching rule segment.

        Args:
            rule: Rule object with segment conditions
            segments_df: DataFrame of binary segment features

        Returns:
            Boolean Series indicating matches
        """
        mask = pd.Series(True, index=segments_df.index)
        for feature, level in rule.segment:
            if feature in segments_df.columns:
                mask &= (segments_df[feature] == level)
            else:
                return pd.Series(False, index=segments_df.index)
        return mask

    def evaluate_candidates(self, candidates, X_val, y_val, rf_model=None):
        """
        Evaluate candidate rules on validation data.

        Args:
            candidates: List of Rule objects from generation
            X_val: Validation features
            y_val: Validation labels
            rf_model: Optional RF model for alignment metrics

        Returns:
            List of evaluated Rule objects with computed metrics
        """
        segments_val = self.segment_builder.assign_segments(X_val)

        # Get RF predictions if model provided
        rf_proba = None
        if rf_model is not None:
            rf_proba = rf_model.predict_proba(X_val)[:, 1]

        evaluated_rules = []

        print(f"\nEvaluating {len(candidates)} candidate rules...")

        for rule in candidates:
            mask = self.match_rule(rule, segments_val)
            n_matches = mask.sum()
            rule.covered_idx = set(np.flatnonzero(mask.values))

            # Filter low-support rules
            if n_matches < self.min_support // 2:
                continue

            y_pred = np.full(n_matches, rule.predicted_class)
            y_true = y_val[mask]

            # ✅ CLASS-SPECIFIC PRECISION
            if rule.predicted_class == 1:
                rule.precision = precision_score(y_true, y_pred, zero_division=0.0)
            else:
                rule.precision = (y_true == 0).sum() / len(y_true) if len(y_true) > 0 else 0.0

            # ✅ CLASS-SPECIFIC RECALL
            if rule.predicted_class == 1:
                total_positives = (y_val == 1).sum()
                true_positives = ((y_val == 1) & mask).sum()
                rule.recall = true_positives / total_positives if total_positives > 0 else 0.0
            else:
                total_negatives = (y_val == 0).sum()
                true_negatives = ((y_val == 0) & mask).sum()
                rule.recall = true_negatives / total_negatives if total_negatives > 0 else 0.0

            # Coverage
            rule.coverage = n_matches / len(X_val)

            # Stability (bootstrap precision variance)
            if rule.predicted_class == 1:
                rule.stability = self._estimate_stability(rule, X_val, y_val)
            else:
                rule.stability = 1.0

            # RF-SPECIFIC METRICS
            if rf_proba is not None:
                # RF Alignment: How confident is RF in this segment?
                # Higher confidence = better alignment
                rf_conf_in_segment = np.abs(rf_proba[mask] - 0.5)
                rule.rf_confidence = rf_conf_in_segment.mean()

                # RF Overlap: What fraction of segment has RF agreeing?
                # (confidence > 0.20 threshold = RF is reasonably sure)
                rule.rf_alignment = (rf_conf_in_segment > 0.20).mean()
            else:
                rule.rf_confidence = 0.5
                rule.rf_alignment = 0.0

            evaluated_rules.append(rule)

        print(f"Evaluated {len(evaluated_rules)} rules "
              f"(filtered {len(candidates) - len(evaluated_rules)} low-coverage)")

        return evaluated_rules

    def _estimate_stability(self, rule, X, y, n_bootstrap=3):
        """
        Estimate rule precision stability via bootstrap sampling.

        High stability = low variance in precision across samples.

        Args:
            rule: Rule to evaluate
            X: Feature data
            y: Labels
            n_bootstrap: Number of bootstrap iterations

        Returns:
            Stability score in [0, 1] (higher = more stable)
        """
        precisions = []
        n = len(X)

        for _ in range(n_bootstrap):
            idx = np.random.choice(n, size=n, replace=True)
            X_sample = X.iloc[idx]
            y_sample = y.iloc[idx] if hasattr(y, 'iloc') else y[idx]

            segments = self.segment_builder.assign_segments(X_sample)
            mask = self.match_rule(rule, segments)

            if mask.sum() < 10:
                continue

            y_pred = np.full(mask.sum(), rule.predicted_class)
            y_true = y_sample[mask]

            if rule.predicted_class == 1:
                prec = precision_score(y_true, y_pred, zero_division=0.0)
            else:
                prec = (y_true == 0).sum() / len(y_true) if len(y_true) > 0 else 0.0

            precisions.append(prec)

        if len(precisions) < 2:
            return 0.0

        mean_prec = np.mean(precisions)
        std_prec = np.std(precisions)

        if mean_prec > 0:
            stability = 1.0 - (std_prec / mean_prec)
            return max(0.0, stability)
        else:
            return 0.0

    def match_rule(self, rule, segments_df):
        """
        Create boolean mask for samples matching rule segment.

        Args:
            rule: Rule object with segment conditions
            segments_df: DataFrame of binary segment features

        Returns:
            Boolean Series indicating matches
        """
        mask = pd.Series(True, index=segments_df.index)
        for feature, level in rule.segment:
            if feature in segments_df.columns:
                mask &= (segments_df[feature] == level)
            else:
                return pd.Series(False, index=segments_df.index)
        return mask

    def evaluate_candidates(self, candidates, X_val, y_val, rf_model=None):
        """
        Evaluate candidate rules on validation data.

        Args:
            candidates: List of Rule objects from generation
            X_val: Validation features
            y_val: Validation labels
            rf_model: Optional RF model for alignment metrics

        Returns:
            List of evaluated Rule objects with computed metrics
        """
        segments_val = self.segment_builder.assign_segments(X_val)

        # Get RF predictions if model provided
        rf_proba = None
        if rf_model is not None:
            rf_proba = rf_model.predict_proba(X_val)[:, 1]

        evaluated_rules = []

        print(f"\nEvaluating {len(candidates)} candidate rules...")

        for rule in candidates:
            mask = self.match_rule(rule, segments_val)
            n_matches = mask.sum()
            rule.covered_idx = set(np.flatnonzero(mask.values))

            # Filter low-support rules
            if n_matches < self.min_support // 2:
                continue

            y_pred = np.full(n_matches, rule.predicted_class)
            y_true = y_val[mask]

            # ✅ CLASS-SPECIFIC PRECISION
            if rule.predicted_class == 1:
                rule.precision = precision_score(y_true, y_pred, zero_division=0.0)
            else:
                rule.precision = (y_true == 0).sum() / len(y_true) if len(y_true) > 0 else 0.0

            # ✅ CLASS-SPECIFIC RECALL
            if rule.predicted_class == 1:
                total_positives = (y_val == 1).sum()
                true_positives = ((y_val == 1) & mask).sum()
                rule.recall = true_positives / total_positives if total_positives > 0 else 0.0
            else:
                total_negatives = (y_val == 0).sum()
                true_negatives = ((y_val == 0) & mask).sum()
                rule.recall = true_negatives / total_negatives if total_negatives > 0 else 0.0

            # Coverage
            rule.coverage = n_matches / len(X_val)

            # Stability (bootstrap precision variance)
            if rule.predicted_class == 1:
                rule.stability = self._estimate_stability(rule, X_val, y_val)
            else:
                rule.stability = 1.0

            # RF-SPECIFIC METRICS
            if rf_proba is not None:
                # RF Alignment: How confident is RF in this segment?
                # Higher confidence = better alignment
                rf_conf_in_segment = np.abs(rf_proba[mask] - 0.5)
                rule.rf_confidence = rf_conf_in_segment.mean()

                # RF Overlap: What fraction of segment has RF agreeing?
                # (confidence > 0.20 threshold = RF is reasonably sure)
                rule.rf_alignment = (rf_conf_in_segment > 0.20).mean()
            else:
                rule.rf_confidence = 0.5
                rule.rf_alignment = 0.0

            evaluated_rules.append(rule)

        print(f"Evaluated {len(evaluated_rules)} rules "
              f"(filtered {len(candidates) - len(evaluated_rules)} low-coverage)")

        # 🔥 SEMANTIC DEDUPLICATION
        before = len(evaluated_rules)
        evaluated_rules = self._deduplicate_rules(evaluated_rules)
        after = len(evaluated_rules)

        if after < before:
            print(f"Deduplicated rules: {before} → {after}")

        return evaluated_rules


    def _estimate_stability(self, rule, X, y, n_bootstrap=3):
        """
        Estimate rule precision stability via bootstrap sampling.

        High stability = low variance in precision across samples.

        Args:
            rule: Rule to evaluate
            X: Feature data
            y: Labels
            n_bootstrap: Number of bootstrap iterations

        Returns:
            Stability score in [0, 1] (higher = more stable)
        """
        precisions = []
        n = len(X)

        for _ in range(n_bootstrap):
            idx = np.random.choice(n, size=n, replace=True)
            X_sample = X.iloc[idx]
            y_sample = y.iloc[idx] if hasattr(y, 'iloc') else y[idx]

            segments = self.segment_builder.assign_segments(X_sample)
            mask = self.match_rule(rule, segments)

            if mask.sum() < 10:
                continue

            y_pred = np.full(mask.sum(), rule.predicted_class)
            y_true = y_sample[mask]

            if rule.predicted_class == 1:
                prec = precision_score(y_true, y_pred, zero_division=0.0)
            else:
                prec = (y_true == 0).sum() / len(y_true) if len(y_true) > 0 else 0.0

            precisions.append(prec)

        if len(precisions) < 2:
            return 0.0

        mean_prec = np.mean(precisions)
        std_prec = np.std(precisions)

        if mean_prec > 0:
            stability = 1.0 - (std_prec / mean_prec)
            return max(0.0, stability)
        else:
            return 0.0


In [139]:
# ============================================================
# CELL 13: ILP RULE SELECTOR
# ============================================================

class ILPRuleSelector:
    """
    Integer Linear Programming solver for optimal rule subset selection.

    Objective:
        max Σ x_r * (precision_r² * recall_r * coverage_r * interpretability_r
                     - λ₁ * rf_uncertainty_r - λ₂ * (1 - rf_alignment_r))

    Constraints:
        - Cardinality: min_rules ≤ Σ x_r ≤ max_rules
        - Precision gates: x_r = 0 if precision_r < threshold
        - Recall gates: x_r = 0 if recall_r < threshold
        - Class balance: min/max rules per class
        - Feature diversity: max uses per feature

    Changes from EBM version:
    - Uses rf_alignment instead of ebm_overlap
    - Uses rf_confidence instead of boundary_ambiguity
    """

    def __init__(
        self,
        min_rules=8,
        max_rules=15,
        min_precision_subscribe=0.50,
        min_precision_not_subscribe=0.50,
        min_recall_subscribe=0.10,
        min_recall_not_subscribe=0.20,
        min_subscribe_rules=3,
        max_subscribe_rules=5,
        min_not_subscribe_rules=3,
        max_not_subscribe_rules=5,
        max_feature_usage=10,
        lambda_rf_uncertainty=0.15,  # Penalty for RF uncertainty
        lambda_rf_misalignment=0.10,  # Penalty for RF disagreement
        recall_weight=1.0
    ):
        self.min_rules = min_rules
        self.max_rules = max_rules
        self.min_precision_subscribe = min_precision_subscribe
        self.min_precision_not_subscribe = min_precision_not_subscribe
        self.min_recall_subscribe = min_recall_subscribe
        self.min_recall_not_subscribe = min_recall_not_subscribe
        self.min_subscribe_rules = min_subscribe_rules
        self.max_subscribe_rules = max_subscribe_rules
        self.min_not_subscribe_rules = min_not_subscribe_rules
        self.max_not_subscribe_rules = max_not_subscribe_rules
        self.max_feature_usage = max_feature_usage
        self.lambda_rf_uncertainty = lambda_rf_uncertainty
        self.lambda_rf_misalignment = lambda_rf_misalignment
        self.recall_weight = recall_weight

    def select_rules(self, candidates):
        """
        Select optimal rule subset via ILP optimization.

        Args:
            candidates: List of evaluated Rule objects

        Returns:
            List of selected Rule objects
        """
        from pulp import LpProblem, LpMaximize, LpVariable, lpSum, LpStatus, PULP_CBC_CMD

        if not candidates:
            print("Warning: No candidates provided to ILP selector.")
            return []

        # ============================================================
        # DEBUG: Show candidates before gating
        # ============================================================
        print("\n" + "="*80)
        print("🔍 ILP DEBUG: CANDIDATES BEFORE GATING")
        print("="*80)

        subscribe_cands = [r for r in candidates if r.predicted_class == 1]
        not_subscribe_cands = [r for r in candidates if r.predicted_class == 0]

        print(f"\nTotal candidates: {len(candidates)}")
        print(f"  SUBSCRIBE candidates: {len(subscribe_cands)}")
        print(f"  NOT_SUBSCRIBE candidates: {len(not_subscribe_cands)}")

        print(f"\n📊 Top 10 SUBSCRIBE candidates:")
        print(f"{'Rule':<6} {'Segment':<50} {'Precision':<10} {'Recall':<10} {'Coverage':<10} {'RF_Align':<10}")
        print("-" * 100)
        for i, r in enumerate(sorted(subscribe_cands, key=lambda x: x.precision, reverse=True)[:10], 1):
            seg_str = ' AND '.join([f"{f}={l}" for f, l in list(r.segment)[:2]])
            if len(r.segment) > 2:
                seg_str += f" ... (+{len(r.segment)-2})"
            rf_align = getattr(r, 'rf_alignment', 0.0)
            print(f"{i:<6} {seg_str:<50} {r.precision:<10.3f} {r.recall:<10.3f} {r.coverage:<10.3f} {rf_align:<10.3f}")

        print(f"\n📊 Top 10 NOT_SUBSCRIBE candidates:")
        print(f"{'Rule':<6} {'Segment':<50} {'Precision':<10} {'Recall':<10} {'Coverage':<10} {'RF_Align':<10}")
        print("-" * 100)
        for i, r in enumerate(sorted(not_subscribe_cands, key=lambda x: x.precision, reverse=True)[:10], 1):
            seg_str = ' AND '.join([f"{f}={l}" for f, l in list(r.segment)[:2]])
            if len(r.segment) > 2:
                seg_str += f" ... (+{len(r.segment)-2})"
            rf_align = getattr(r, 'rf_alignment', 0.0)
            print(f"{i:<6} {seg_str:<50} {r.precision:<10.3f} {r.recall:<10.3f} {r.coverage:<10.3f} {rf_align:<10.3f}")

        # ============================================================
        # Setup ILP problem
        # ============================================================
        max_seen = -1
        for r in candidates:
            if hasattr(r, "covered_idx") and r.covered_idx and len(r.covered_idx) > 0:
                max_seen = max(max_seen, max(r.covered_idx))

        N = max_seen + 1
        if N <= 0:
            print("Warning: Could not infer validation size.")
            return []

        prob = LpProblem("GLASS_BRW_RF_Selection", LpMaximize)
        x = {r.rule_id: LpVariable(f"x_{r.rule_id}", cat='Binary') for r in candidates}

        # ============================================================
        # OBJECTIVE: RF-ALIGNED QUALITY
        # ============================================================
        objective_terms = []
        for r in candidates:
            # Base quality
            quality = (r.precision ** 2) * r.recall * r.coverage * r.interpretability

            # RF alignment bonus/penalty
            rf_uncertainty = 1.0 - getattr(r, 'rf_confidence', 0.5)
            rf_misalignment = 1.0 - getattr(r, 'rf_alignment', 0.0)

            # Penalize RF-uncertain or misaligned rules
            adjusted_quality = (
                quality
                - self.lambda_rf_uncertainty * rf_uncertainty
                - self.lambda_rf_misalignment * rf_misalignment
            )

            objective_terms.append(x[r.rule_id] * adjusted_quality)

        prob += lpSum(objective_terms), "RF_Aligned_Objective"

        # ============================================================
        # PRECISION/RECALL GATES
        # ============================================================
        print("\n" + "="*80)
        print("🔍 APPLYING PRECISION/RECALL GATES")
        print("="*80)
        print(f"\nGate thresholds:")
        print(f"  SUBSCRIBE:     precision ≥ {self.min_precision_subscribe:.3f}, recall ≥ {self.min_recall_subscribe:.3f}")
        print(f"  NOT_SUBSCRIBE: precision ≥ {self.min_precision_not_subscribe:.3f}, recall ≥ {self.min_recall_not_subscribe:.3f}")

        subscribe_valid = []
        subscribe_rejected = []
        not_subscribe_valid = []
        not_subscribe_rejected = []

        for r in candidates:
            if r.predicted_class == 1:
                if r.precision >= self.min_precision_subscribe and r.recall >= self.min_recall_subscribe:
                    subscribe_valid.append(r)
                else:
                    subscribe_rejected.append(r)
                    prob += (x[r.rule_id] == 0, f"Gate_Sub_{r.rule_id}")
            else:
                if r.precision >= self.min_precision_not_subscribe and r.recall >= self.min_recall_not_subscribe:
                    not_subscribe_valid.append(r)
                else:
                    not_subscribe_rejected.append(r)
                    prob += (x[r.rule_id] == 0, f"Gate_NotSub_{r.rule_id}")

        print(f"\n✅ PASSED GATES:")
        print(f"  SUBSCRIBE:     {len(subscribe_valid)}/{len(subscribe_cands)}")
        print(f"  NOT_SUBSCRIBE: {len(not_subscribe_valid)}/{len(not_subscribe_cands)}")

        print(f"\n❌ REJECTED BY GATES:")
        print(f"  SUBSCRIBE:     {len(subscribe_rejected)}/{len(subscribe_cands)}")
        print(f"  NOT_SUBSCRIBE: {len(not_subscribe_rejected)}/{len(not_subscribe_cands)}")

               # Show rejection reasons for top SUBSCRIBE rules
        if len(subscribe_rejected) > 0:
            print(f"\n❌ Top 5 REJECTED SUBSCRIBE rules:")
            for i, r in enumerate(
                sorted(subscribe_rejected, key=lambda x: x.precision, reverse=True)[:5], 1
            ):
                seg_str = ' AND '.join([f"{f}={l}" for f, l in list(r.segment)[:2]])
                if len(r.segment) > 2:
                    seg_str += f" ... (+{len(r.segment)-2})"

                fail_reasons = []
                if r.precision < self.min_precision_subscribe:
                    fail_reasons.append(
                        f"precision {r.precision:.3f} < {self.min_precision_subscribe:.3f}"
                    )
                if r.recall < self.min_recall_subscribe:
                    fail_reasons.append(
                        f"recall {r.recall:.3f} < {self.min_recall_subscribe:.3f}"
                    )

                print(f"  {i}. {seg_str}")
                print(f"     FAILED: {', '.join(fail_reasons)}")

        # ============================================================
        # BUILD VALID RULE SET (ONLY RULES THAT PASSED GATES)
        # ============================================================
        valid = subscribe_valid + not_subscribe_valid
        print(f"\nTotal valid after gates: {len(valid)}")

        if len(valid) == 0:
            print("⚠️ No rules passed gates. Returning empty selection.")
            return []

        # ============================================================
        # SETUP ILP OVER VALID RULES ONLY
        # ============================================================
        prob = LpProblem("GLASS_BRW_RF_Selection", LpMaximize)
        x = {r.rule_id: LpVariable(f"x_{r.rule_id}", cat="Binary") for r in valid}

        # ============================================================
        # OBJECTIVE
        # ============================================================
        prob += lpSum(
            x[r.rule_id]
            * (
                (r.precision ** 2)
                * r.recall
                * r.coverage
                * r.interpretability
                - self.lambda_rf_uncertainty * (1.0 - getattr(r, "rf_confidence", 0.5))
                - self.lambda_rf_misalignment * (1.0 - getattr(r, "rf_alignment", 0.0))
            )
            for r in valid
        )

        # ============================================================
        # CARDINALITY CONSTRAINTS (CLAMPED)
        # ============================================================
        actual_min_sub = min(
            self.min_subscribe_rules,
            len(subscribe_valid),
            self.max_subscribe_rules
        )
        actual_min_not_sub = min(
            self.min_not_subscribe_rules,
            len(not_subscribe_valid),
            self.max_not_subscribe_rules
        )

        max_possible = min(
            self.max_rules,
            self.max_subscribe_rules + self.max_not_subscribe_rules,
            len(valid)
        )
        actual_min_rules = min(self.min_rules, len(valid), max_possible)

        prob += lpSum(x[r.rule_id] for r in valid) >= actual_min_rules
        prob += lpSum(x[r.rule_id] for r in valid) <= min(self.max_rules, len(valid))

        if subscribe_valid:
            prob += lpSum(x[r.rule_id] for r in subscribe_valid) >= actual_min_sub
            prob += lpSum(x[r.rule_id] for r in subscribe_valid) <= min(
                self.max_subscribe_rules, len(subscribe_valid)
            )

        if not_subscribe_valid:
            prob += lpSum(x[r.rule_id] for r in not_subscribe_valid) >= actual_min_not_sub
            prob += lpSum(x[r.rule_id] for r in not_subscribe_valid) <= min(
                self.max_not_subscribe_rules, len(not_subscribe_valid)
            )

        # ============================================================
        # FEATURE DIVERSITY (VALID ONLY)
        # ============================================================
        feature_usage = defaultdict(list)
        for r in valid:
            for feature, _ in r.segment:
                feature_usage[feature].append(r.rule_id)

        for feature, rule_ids in feature_usage.items():
            prob += lpSum(x[rid] for rid in rule_ids) <= self.max_feature_usage

        # ============================================================
        # SOLVE ILP
        # ============================================================
        print("\n" + "=" * 80)
        print("Solving RF-aligned ILP...")
        solver = PULP_CBC_CMD(msg=0, timeLimit=300)
        prob.solve(solver)

        status = LpStatus[prob.status]
        print(f"ILP Status: {status}")
        print("=" * 80)

        # ============================================================
        # FALLBACK IF INFEASIBLE
        # ============================================================
        if status != "Optimal":
            print("\n⚠️ ILP FAILED - Using greedy fallback")

            subscribe_sorted = sorted(
                subscribe_valid,
                key=lambda r: (r.precision ** 2) * r.recall * r.coverage * getattr(r, "rf_alignment", 0.5),
                reverse=True
            )
            not_subscribe_sorted = sorted(
                not_subscribe_valid,
                key=lambda r: (r.precision ** 2) * r.recall * r.coverage * getattr(r, "rf_alignment", 0.5),
                reverse=True
            )

            selected_rules = (
                subscribe_sorted[:self.max_subscribe_rules]
                + not_subscribe_sorted[:self.max_not_subscribe_rules]
            )

            print(f"   Selected {len(subscribe_sorted[:self.max_subscribe_rules])} SUBSCRIBE rules")
            print(f"   Selected {len(not_subscribe_sorted[:self.max_not_subscribe_rules])} NOT_SUBSCRIBE rules")
            print(f"   Total: {len(selected_rules)} rules")

            return selected_rules

        # ============================================================
        # EXTRACT SOLUTION
        # ============================================================
        selected_rules = [
            r for r in valid
            if x[r.rule_id].varValue is not None and x[r.rule_id].varValue > 0.5
        ]

        print(f"\nSelected {len(selected_rules)} rules via ILP")
        return selected_rules

In [143]:
# ============================================================
# CELL 14: GLASS-BRW RF-BASED RULE EXTRACTION - PART 4
# Main Classifier with First-Match-Wins Inference
# ============================================================

import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score

class GLASS_BRW:
    """
    GLASS-BRW: Precision-Gated Rule Ensemble with Abstention.
    """

    def __init__(
        self,
        mode="strict",
        min_precision_subscribe=0.50,
        min_precision_not_subscribe=0.50,
        min_recall_subscribe=0.10,
        min_recall_not_subscribe=0.10,
        min_support=25,
        max_complexity=3,
        min_rules=8,
        max_rules=10,
        max_subscribe_rules=5,
        max_not_subscribe_rules=5,
        rf_model=None
    ):
        self.mode = mode
        self.min_precision_subscribe = min_precision_subscribe
        self.min_precision_not_subscribe = min_precision_not_subscribe
        self.min_recall_subscribe = min_recall_subscribe
        self.min_recall_not_subscribe = min_recall_not_subscribe
        self.min_support = min_support
        self.max_complexity = max_complexity
        self.min_rules = min_rules
        self.max_rules = max_rules
        self.max_subscribe_rules = max_subscribe_rules
        self.max_not_subscribe_rules = max_not_subscribe_rules

        # Components
        self.segment_builder = BankSegmentBuilder()

        self.rule_generator = RuleGenerator(
            min_support=min_support,
            max_complexity=max_complexity,
            mode=mode,
            rf_model=rf_model,
            min_precision_subscribe=min_precision_subscribe,
            min_precision_not_subscribe=min_precision_not_subscribe,
            min_recall_subscribe=min_recall_subscribe,
            min_recall_not_subscribe=min_recall_not_subscribe
        )

        self.rule_evaluator = RuleEvaluator(self.segment_builder, min_support)

        self.ilp_selector = ILPRuleSelector(
            min_rules=min_rules,
            max_rules=max_rules,
            min_precision_subscribe=min_precision_subscribe,
            min_precision_not_subscribe=min_precision_not_subscribe,
            min_recall_subscribe=min_recall_subscribe,
            min_recall_not_subscribe=min_recall_not_subscribe,
            min_subscribe_rules=3,
            max_subscribe_rules=max_subscribe_rules,
            min_not_subscribe_rules=3,
            max_not_subscribe_rules=max_not_subscribe_rules
        )

        # Fitted attributes
        self.rules = None
        self.rf_model = rf_model
        self.is_fitted = False

    def fit(self, X_train, y_train, X_val, y_val, rf_model=None):
        """
        Train GLASS-BRW using RF-guided rule generation and ILP optimization.
        """
        print("="*80)
        print("GLASS-BRW Training Pipeline (RF-Powered)")
        print("="*80)
        print(f"Mode: {self.mode}")
        print(f"Max complexity: {self.max_complexity}")
        print(f"Precision gates: SUB ≥ {self.min_precision_subscribe:.2f}, NOT_SUB ≥ {self.min_precision_not_subscribe:.2f}")
        print(f"Recall gates: SUB ≥ {self.min_recall_subscribe:.2f}, NOT_SUB ≥ {self.min_recall_not_subscribe:.2f}")

        # Update RF model if provided
        if rf_model is not None:
            self.rf_model = rf_model
            self.rule_generator.rf_model = rf_model

        # Convert to pandas
        if not isinstance(X_train, pd.DataFrame):
            X_train = pd.DataFrame(X_train)
        if not isinstance(X_val, pd.DataFrame):
            X_val = pd.DataFrame(X_val)
        if not isinstance(y_train, (pd.Series, np.ndarray)):
            y_train = np.array(y_train)
        if not isinstance(y_val, (pd.Series, np.ndarray)):
            y_val = np.array(y_val)

        # ============================================================
        # PHASE 1: Assign customer segments
        # ============================================================
        print("\n" + "="*80)
        print("PHASE 1: Assigning customer segments")
        print("="*80)
        segments_train = self.segment_builder.assign_segments(X_train)
        print(f"  Training samples: {len(segments_train)}")
        print(f"  Segment features: {len(segments_train.columns)}")
        print(f"  Feature tiers: {self.mode}")

        # ============================================================
        # PHASE 2: Generate candidate rules (RF-guided)
        # ============================================================
        print("\n" + "="*80)
        print("PHASE 2: Generating candidate rules (RF-guided)")
        print("="*80)
        candidates = self.rule_generator.generate_candidates(segments_train, y_train)

        print(f"\nCandidate summary:")
        sub_cands = [r for r in candidates if r.predicted_class == 1]
        not_sub_cands = [r for r in candidates if r.predicted_class == 0]
        print(f"  SUBSCRIBE: {len(sub_cands)}")
        print(f"  NOT_SUBSCRIBE: {len(not_sub_cands)}")

        # ============================================================
        # PHASE 3: Evaluate on validation
        # ============================================================
        print("\n" + "="*80)
        print("PHASE 3: Evaluating rule quality on validation set")
        print("="*80)
        evaluated_candidates = self.rule_evaluator.evaluate_candidates(
            candidates, X_val, y_val, rf_model=self.rf_model
        )

        print(f"\nEvaluated candidates: {len(evaluated_candidates)}")

        # ============================================================
        # PHASE 4: ILP selection
        # ============================================================
        print("\n" + "="*80)
        print("PHASE 4: Solving ILP for optimal rule subset")
        print("="*80)
        selected_rules = self.ilp_selector.select_rules(evaluated_candidates)

        if not selected_rules:
            print("\n⚠️  WARNING: No rules selected! Relaxing constraints...")
            # Fallback: take top rules by precision
            sub_rules = sorted([r for r in evaluated_candidates if r.predicted_class == 1],
                             key=lambda r: r.precision, reverse=True)[:5]
            not_sub_rules = sorted([r for r in evaluated_candidates if r.predicted_class == 0],
                                  key=lambda r: r.precision, reverse=True)[:5]
            selected_rules = sub_rules + not_sub_rules

        # ============================================================
        # PHASE 5: Order rules (SUBSCRIBE → NOT_SUBSCRIBE, precision-sorted)
        # ============================================================
        print("\n" + "="*80)
        print("PHASE 5: Ordering rules (SUBSCRIBE → NOT_SUBSCRIBE)")
        print("="*80)

        subscribe_rules = sorted(
            [r for r in selected_rules if r.predicted_class == 1],
            key=lambda r: r.precision,
            reverse=True
        )
        not_subscribe_rules = sorted(
            [r for r in selected_rules if r.predicted_class == 0],
            key=lambda r: r.precision,
            reverse=True
        )
        self.rules = subscribe_rules + not_subscribe_rules

        # ============================================================
        # Print final ruleset
        # ============================================================
        print("\n" + "="*80)
        print("✅ FINAL RULESET (ordered by precision)")
        print("="*80)

        for i, rule in enumerate(self.rules, 1):
            segment_str = ' AND '.join([f"{f}={l}" for f, l in rule.segment])
            class_str = "SUBSCRIBE" if rule.predicted_class == 1 else "NOT_SUB"
            rf_align = getattr(rule, 'rf_alignment', 0.0)

            print(f"\n{i}. [{class_str:9s}] {segment_str}")
            print(f"   Precision: {rule.precision:.1%}, Recall: {rule.recall:.1%}, "
                  f"Coverage: {rule.coverage:.1%}, Complexity: {rule.complexity}")
            print(f"   RF Alignment: {rf_align:.1%}")

        print("\n" + "="*80)
        print(f"Total rules: {len(self.rules)}")
        print(f"  SUBSCRIBE: {len(subscribe_rules)}")
        print(f"  NOT_SUBSCRIBE: {len(not_subscribe_rules)}")
        print("="*80)

        self.is_fitted = True
        return self

    def predict(self, X):
        """
        Predict using first-match-wins with explicit abstention.
        """
        if not self.is_fitted:
            raise ValueError("Model must be fitted before prediction")

        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        n = len(X)
        predictions = np.full(n, -1, dtype=int)
        explanations = [None] * n
        confidences = np.zeros(n)

        # Assign segments
        segments = self.segment_builder.assign_segments(X)

        # First-match-wins
        for i in range(n):
            for rule in self.rules:
                # Check if sample matches rule segment
                matches = True
                for feature, level in rule.segment:
                    if feature not in segments.columns:
                        matches = False
                        break
                    if segments.iloc[i][feature] != level:
                        matches = False
                        break

                if not matches:
                    continue

                # Optional: enforce precision floor at execution
                if rule.predicted_class == 1 and rule.precision < self.min_precision_subscribe:
                    continue
                if rule.predicted_class == 0 and rule.precision < self.min_precision_not_subscribe:
                    continue

                # Match found - make prediction
                predictions[i] = rule.predicted_class
                confidences[i] = rule.precision
                explanations[i] = self._format_explanation(rule)
                break

            # No rule matched → abstain
            if predictions[i] == -1:
                explanations[i] = "ABSTAIN: No high-precision segment rule applies"
                confidences[i] = 0.0

        # Store for later access
        self.last_predictions = predictions
        self.last_explanations = explanations
        self.last_confidences = confidences

        return predictions, explanations, confidences

    def predict_proba(self, X):
        """
        Return probability estimates [P(Not_Subscribe), P(Subscribe)].
        """
        predictions, _, confidences = self.predict(X)
        n = len(X)
        probas = np.zeros((n, 2))

        for i in range(n):
            if predictions[i] == -1:  # ABSTAIN
                probas[i] = [0.88, 0.12]
            elif predictions[i] == 1:  # SUBSCRIBE
                probas[i] = [1 - confidences[i], confidences[i]]
            else:  # NOT_SUBSCRIBE
                probas[i] = [confidences[i], 1 - confidences[i]]

        return probas

    def _format_explanation(self, rule):
        """
        Generate human-readable explanation for a rule.
        """
        conditions = [f"{feat}={level}" for feat, level in rule.segment]
        segment_str = ' AND '.join(conditions)
        class_name = "SUBSCRIBE" if rule.predicted_class == 1 else "NOT_SUBSCRIBE"
        return f"Segment: {segment_str} → {class_name} ({rule.precision:.1%} precision)"

    def explain_prediction(self, X, sample_idx=0):
        """
        Get explanation for a specific prediction.
        """
        if not hasattr(self, 'last_explanations'):
            self.predict(X)
        if sample_idx < len(self.last_explanations):
            return self.last_explanations[sample_idx]
        return "No explanation available"

    def get_coverage_stats(self, X, y):
        """
        Compute coverage and performance statistics.
        """
        predictions, _, confidences = self.predict(X)

        covered = (predictions != -1)
        coverage_rate = covered.mean()
        abstention_rate = 1 - coverage_rate

        # Performance on covered samples
        if covered.sum() > 0:
            covered_precision = precision_score(y[covered], predictions[covered])
            covered_recall = recall_score(y[covered], predictions[covered])
        else:
            covered_precision = 0.0
            covered_recall = 0.0

        # Class-specific coverage
        subscribe_covered = covered & (predictions == 1)
        not_subscribe_covered = covered & (predictions == 0)

        stats = {
            'coverage_rate': coverage_rate,
            'abstention_rate': abstention_rate,
            'covered_precision': covered_precision,
            'covered_recall': covered_recall,
            'subscribe_coverage': subscribe_covered.mean(),
            'not_subscribe_coverage': not_subscribe_covered.mean(),
            'avg_confidence': confidences[covered].mean() if covered.sum() > 0 else 0.0,
            'n_samples': len(X),
            'n_covered': covered.sum(),
            'n_abstained': (~covered).sum()
        }

        return stats

    def get_rule_summary(self):
        """
        Get summary of selected rules.
        """
        if not self.is_fitted:
            raise ValueError("Model must be fitted first")

        summary = []
        for i, rule in enumerate(self.rules, 1):
            segment_str = ' AND '.join([f"{f}={l}" for f, l in rule.segment])
            class_name = "SUBSCRIBE" if rule.predicted_class == 1 else "NOT_SUBSCRIBE"
            rf_align = getattr(rule, 'rf_alignment', 0.0)

            summary.append({
                'rule_id': i,
                'class': class_name,
                'segment': segment_str,
                'precision': rule.precision,
                'recall': rule.recall,
                'coverage': rule.coverage,
                'complexity': rule.complexity,
                'rf_alignment': rf_align,
                'interpretability': rule.interpretability
            })

        return pd.DataFrame(summary)

In [144]:
# ============================================================
# CELL 15: COMPLETE GLASS-BRW PIPELINE (RF-POWERED)
# USING GLOBAL SPLIT - ENGINEER FEATURES AFTER SPLIT
# ============================================================
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. USE GLOBAL SPLIT FIRST, THEN ENGINEER
# ============================================================
print("\n" + "="*80)
print("📊 STEP 1: USING GLOBAL TRAIN/TEST SPLIT")
print("="*80)

# Get the split from df_proc
X_train_raw = GLOBAL_SPLIT['X_train'].copy()
X_test_raw = GLOBAL_SPLIT['X_test'].copy()
y_train = GLOBAL_SPLIT['y_train'].copy()
y_test = GLOBAL_SPLIT['y_test'].copy()

print(f"\n✅ Using Global Split:")
print(f"   Train: {len(X_train_raw):5,} | Pos: {y_train.mean():.2%}")
print(f"   Test:  {len(X_test_raw):5,} | Pos: {y_test.mean():.2%}")

# ============================================================
# 2. ENGINEER FEATURES ON TRAIN AND TEST SEPARATELY
# ============================================================
print("\n" + "="*80)
print("🔍 STEP 2: FEATURE ENGINEERING")
print("="*80)

# Add back the target for feature engineering
train_with_y = X_train_raw.copy()
train_with_y['y'] = y_train

test_with_y = X_test_raw.copy()
test_with_y['y'] = y_test

# Engineer features
df_train_eng = engineer_features_bank(train_with_y)
df_test_eng = engineer_features_bank(test_with_y)

# Feature list
FEATURES = [
    "dur_low", "dur_mid", "dur_high",
    "month_early", "month_late_spring", "month_summer", "month_fall", "month_year_end",
    "engagement_low", "fatigue_low", "poutcome_success",
    "housing", "pdays_never"
]

# Extract features
X_train = df_train_eng[FEATURES].copy()
X_test = df_test_eng[FEATURES].copy()

print(f"\n✅ Feature Engineering Complete:")
print(f"   Features: {len(FEATURES)}")
print(f"   Train shape: {X_train.shape}")
print(f"   Test shape: {X_test.shape}")

# ============================================================
# 3. TRAIN RANDOM FOREST
# ============================================================
print("\n" + "="*80)
print("🌲 STEP 3: TRAINING RANDOM FOREST")
print("="*80)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_split=50,
    min_samples_leaf=25,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

rf_model.fit(X_train, y_train)

rf_test_pred = rf_model.predict(X_test)
rf_test_proba = rf_model.predict_proba(X_test)[:, 1]

print(f"\n📊 RF Performance on Test:")
print(f"   Accuracy:  {accuracy_score(y_test, rf_test_pred):.3f}")
print(f"   Precision: {precision_score(y_test, rf_test_pred, zero_division=0):.3f}")
print(f"   Recall:    {recall_score(y_test, rf_test_pred, zero_division=0):.3f}")

# ============================================================
# 4. TRAIN GLASS-BRW
# ============================================================
print("\n" + "="*80)
print("🚀 STEP 4: TRAINING GLASS-BRW")
print("="*80)

glass = GLASS_BRW(
    mode="strict",  # Use Tier 1 + 2 + 3 features
    min_precision_subscribe=0.10,
    min_precision_not_subscribe=0.10,
    min_recall_subscribe=0.05,
    min_recall_not_subscribe=0.50,
    min_support=25,
    max_complexity=3,
    min_rules=8,
    max_rules=10,
    max_subscribe_rules=5,
    max_not_subscribe_rules=3,
    rf_model=rf_model  # Pass trained RF for alignment
)


glass.fit(X_train, y_train, X_train, y_train, rf_model=rf_model)

# ============================================================
# 5. GENERATE PREDICTIONS ON TRAIN AND TEST
# ============================================================
print("\n" + "="*80)
print("📊 STEP 5: GENERATING PREDICTIONS")
print("="*80)

# 🔥 FIXED: Use predict_proba for proper probability estimates
print("Generating predictions with predict_proba...")
train_proba = glass.predict_proba(X_train)
test_proba = glass.predict_proba(X_test)

# Also get regular predictions for coverage stats
train_pred, train_expl, train_conf = glass.predict(X_train)
test_pred, test_expl, test_conf = glass.predict(X_test)

train_out = {
    "pred": train_pred,
    "confidence": train_conf,
    "covered": (train_pred != -1),
    "abstained": (train_pred == -1),
    "explanations": train_expl,
    "proba": train_proba  # Add probability array
}

test_out = {
    "pred": test_pred,
    "confidence": test_conf,
    "covered": (test_pred != -1),
    "abstained": (test_pred == -1),
    "explanations": test_expl,
    "proba": test_proba  # Add probability array
}

print(f"   Train: {train_out['covered'].sum():5,}/{len(train_pred)} ({train_out['covered'].mean():.1%})")
print(f"   Test:  {test_out['covered'].sum():5,}/{len(test_pred)} ({test_out['covered'].mean():.1%})")

# ============================================================
# 6. EVALUATE ON TEST
# ============================================================
print("\n" + "="*80)
print("📊 STEP 6: TEST SET EVALUATION")
print("="*80)

covered = test_out['covered']
preds_for_metrics = test_pred.copy()
preds_for_metrics[preds_for_metrics == -1] = 0

print(f"\n📊 Overall Performance:")
print(f"   Accuracy:  {accuracy_score(y_test, preds_for_metrics):.3f}")
print(f"   Precision: {precision_score(y_test, preds_for_metrics, zero_division=0):.3f}")
print(f"   Recall:    {recall_score(y_test, preds_for_metrics, zero_division=0):.3f}")

if covered.sum() > 0:
    print(f"\n📊 Covered Cases Only:")
    print(f"   Samples:   {covered.sum():,}")
    print(f"   Precision: {precision_score(y_test[covered], test_pred[covered], zero_division=0):.3f}")
    print(f"   Recall:    {recall_score(y_test[covered], test_pred[covered], zero_division=0):.3f}")
    print(f"   Confidence: {test_conf[covered].mean():.3f}")

# ============================================================
# 7. SAVE META-SAFE ARTIFACTS - FIXED VERSION
# ============================================================
print("\n" + "="*80)
print("💾 STEP 7: SAVING META-SAFE ARTIFACTS")
print("="*80)

from datetime import datetime
import joblib

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 🔥 FIX: Get probabilities from predict_proba, not confidence
print("Generating probabilities with predict_proba...")
train_proba_final = glass.predict_proba(X_train)
test_proba_final = glass.predict_proba(X_test)

print(f"train_proba_final shape: {train_proba_final.shape}")
print(f"test_proba_final shape: {test_proba_final.shape}")

# Debug: Show first 5 samples
print("\nFirst 5 train probabilities (should be [P(NOT_SUB), P(SUB)]):")
for i in range(5):
    print(f"  Sample {i}: {train_proba_final[i]} (abstained: {train_pred[i] == -1})")

glass_bundle = {
    "features": FEATURES,
    "train_idx": GLOBAL_SPLIT["train_idx"],
    "test_idx": GLOBAL_SPLIT["test_idx"],

    "train_pred": train_out["pred"],
    "train_confidence": train_out["confidence"],
    "train_covered": train_out["covered"],
    "train_abstained": train_out["abstained"],

    "test_pred": test_out["pred"],
    "test_confidence": test_out["confidence"],
    "test_covered": test_out["covered"],
    "test_abstained": test_out["abstained"],

    # 🔥 FIX: Use predict_proba output directly
    "train_proba": train_proba_final,
    "test_proba": test_proba_final,

    "rules": glass.get_rule_summary(),

    "config": {
        "mode": glass.mode,
        "min_precision_subscribe": glass.min_precision_subscribe,
        "min_precision_not_subscribe": glass.min_precision_not_subscribe,
        "min_recall_subscribe": glass.min_recall_subscribe,
        "min_recall_not_subscribe": glass.min_recall_not_subscribe,
        "min_support": glass.min_support,
        "max_complexity": glass.max_complexity
    }
}

joblib.dump(glass_bundle, f"./models/glass_brw_bundle_fixed_{timestamp}.joblib")

print(f"\n✅ SAVED: ./models/glass_brw_bundle_fixed_{timestamp}.joblib")

# 🔥 DEBUG: Verify what was saved
print(f"\n🔍 VERIFICATION: What was saved")
print(f"   train_proba shape: {glass_bundle['train_proba'].shape}")
print(f"   Column 0 should be P(NOT_SUB)")
print(f"   Column 1 should be P(SUB)")

# Check a few samples
print(f"\nSample probabilities (first 10):")
print(f"{'Sample':<8} {'P(NOT_SUB)':<12} {'P(SUB)':<12} {'Abstained?'}")
print("-" * 50)

for i in range(min(10, len(glass_bundle['train_proba']))):
    p_not_sub = glass_bundle['train_proba'][i, 0]
    p_sub = glass_bundle['train_proba'][i, 1]
    abstained = glass_bundle['train_pred'][i] == -1
    print(f"{i:<8} {p_not_sub:<12.6f} {p_sub:<12.6f} {abstained}")

print("\n" + "="*80)
print("✅ GLASS-BRW COMPLETE (FIXED PROBABILITIES)")
print("="*80)


📊 STEP 1: USING GLOBAL TRAIN/TEST SPLIT

✅ Using Global Split:
   Train: 36,168 | Pos: 11.70%
   Test:  9,043 | Pos: 11.70%

🔍 STEP 2: FEATURE ENGINEERING
🔍 Engineering RF-aligned features...
✅ RF-driven semantic states created
✅ Engineered 13 RF-aligned features (+ y)
🔍 Engineering RF-aligned features...
✅ RF-driven semantic states created
✅ Engineered 13 RF-aligned features (+ y)

✅ Feature Engineering Complete:
   Features: 13
   Train shape: (36168, 13)
   Test shape: (9043, 13)

🌲 STEP 3: TRAINING RANDOM FOREST

📊 RF Performance on Test:
   Accuracy:  0.715
   Precision: 0.276
   Recall:    0.888

🚀 STEP 4: TRAINING GLASS-BRW
GLASS-BRW Training Pipeline (RF-Powered)
Mode: strict
Max complexity: 3
Precision gates: SUB ≥ 0.10, NOT_SUB ≥ 0.10
Recall gates: SUB ≥ 0.05, NOT_SUB ≥ 0.50

PHASE 1: Assigning customer segments
  Training samples: 36168
  Segment features: 13
  Feature tiers: strict

PHASE 2: Generating candidate rules (RF-guided)

Top 10 RF features by importance:
  dur_lo

In [24]:
!pip install interpret -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 62.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 106.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 19.9 MB/s eta 0:00:00


In [38]:
# ============================================================
# CELL 11: EBM MODEL - COMPREHENSIVE ANALYSIS & ENSEMBLE PREP
# ============================================================
print("=" * 80)
print("🌳 CELL 11: EXPLAINABLE BOOSTING MACHINE (EBM) - FULL ANALYSIS")
print("=" * 80)

# INSTALLATION COMMAND - RUN THIS FIRST IF NEEDED
print("\n💻 INSTALL REQUIRED PACKAGE:")
print("!pip install interpret -q")
print("\nIf already installed, continue with the analysis below.\n")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pickle
import joblib
import os
import warnings
from datetime import datetime
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                           f1_score, roc_auc_score, roc_curve, auc,
                           confusion_matrix, classification_report)
from sklearn.preprocessing import StandardScaler
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.feature_selection import mutual_info_classif
warnings.filterwarnings('ignore')

# Set consistent color scheme
COLOR_PALETTE = {
    'primary': '#1f77b4',      # Blue
    'secondary': '#ff7f0e',    # Orange
    'positive': '#2ca02c',     # Green
    'negative': '#d62728',     # Red
    'neutral': '#7f7f7f',      # Gray
    'highlight': '#9467bd',    # Purple
    'background': '#f0f0f0'
}

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette(sns.color_palette(list(COLOR_PALETTE.values())))

# --------------------------------------------------
# 11.1 DATA PREPARATION (GLOBAL SPLIT)
# --------------------------------------------------
print("\n🔧 11.1 DATA PREPARATION (GLOBAL SPLIT)")
print("-" * 50)

X = df_proc.drop(columns=["y"])
y = df_proc["y"]

X_train = X.loc[GLOBAL_SPLIT["train_idx"]]
y_train = y.loc[GLOBAL_SPLIT["train_idx"]]

X_test  = X.loc[GLOBAL_SPLIT["test_idx"]]
y_test  = y.loc[GLOBAL_SPLIT["test_idx"]]

print(f"\n✓ GLOBAL SPLIT APPLIED:")
print(f"  Train: {X_train.shape} | Pos rate: {y_train.mean():.3f}")
print(f"  Test:  {X_test.shape} | Pos rate: {y_test.mean():.3f}")

# 10-fold CV (TRAIN ONLY — correct)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
print(f"\n✓ 10-fold CV configured on TRAIN only")

# --------------------------------------------------
# 11.2 EBM MODEL TRAINING WITH CROSS-VALIDATION
# --------------------------------------------------
print("\n🤖 11.2 EBM MODEL TRAINING & CROSS-VALIDATION")
print("-" * 50)

# Initialize EBM model
ebm_model = ExplainableBoostingClassifier(
    random_state=42,
    max_bins=256,
    max_interaction_bins=32,
    interactions=10,
    learning_rate=0.01,
    validation_size=0.1,
    n_jobs=-1
)

print("✓ EBM model initialized with configuration:")
print(f"  • max_bins: 256")
print(f"  • interactions: 10")
print(f"  • learning_rate: 0.01")
print(f"  • validation_size: 0.1")

# Perform cross-validation
print("\n🔍 Performing 10-fold cross-validation...")
cv_scores = cross_val_score(
    ebm_model, X_train, y_train,
    cv=cv, scoring='roc_auc',
    n_jobs=-1
)

print(f"\n✓ Cross-validation complete!")
print(f"CV ROC-AUC scores: {cv_scores}")
print(f"Mean CV ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

# Train final model on full training data
print("\n🏋️ Training final EBM model on full training set...")
ebm_model.fit(X_train, y_train)

# Make predictions
y_pred_ebm = ebm_model.predict(X_test)
y_pred_proba_ebm = ebm_model.predict_proba(X_test)[:, 1]

# Calculate comprehensive metrics
metrics_ebm = {
    'Accuracy': accuracy_score(y_test, y_pred_ebm),
    'Precision': precision_score(y_test, y_pred_ebm, average='binary'),
    'Recall': recall_score(y_test, y_pred_ebm, average='binary'),
    'F1-Score': f1_score(y_test, y_pred_ebm, average='binary'),
    'ROC-AUC': roc_auc_score(y_test, y_pred_proba_ebm),
    'CV_ROC_AUC_mean': cv_scores.mean(),
    'CV_ROC_AUC_std': cv_scores.std()
}

print("\n✅ EBM Model Performance Summary:")
print("="*60)
for metric, value in metrics_ebm.items():
    if 'CV' not in metric:
        print(f"{metric}: {value:.4f}")

print(f"\nCross-Validation Performance:")
print(f"Mean ROC-AUC: {cv_scores.mean():.4f}")
print(f"Std ROC-AUC: {cv_scores.std():.4f}")
print(f"Range: [{cv_scores.min():.4f}, {cv_scores.max():.4f}]")

# --------------------------------------------------
# 11.3 EBM FEATURE IMPORTANCE & GLOBAL EXPLANATIONS
# --------------------------------------------------
print("\n📊 11.3 EBM FEATURE IMPORTANCE & GLOBAL EXPLANATIONS")
print("-" * 50)

# Get feature importance from EBM
try:
    importance_scores = ebm_model.feature_importances_
except:
    # Fallback to permutation importance if standard importances not available
    from sklearn.inspection import permutation_importance
    perm_importance = permutation_importance(ebm_model, X_test, y_test,
                                           n_repeats=10, random_state=42)
    importance_scores = perm_importance.importances_mean

feature_importance_ebm = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importance_scores,
    'Abs_Importance': np.abs(importance_scores)
}).sort_values('Abs_Importance', ascending=False)

print("\nTop 15 Most Important Features:")
print(feature_importance_ebm.head(15).to_string(index=False))

# Visualize feature importance
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Bar plot of feature importance
top_n = 15
top_features_ebm = feature_importance_ebm.head(top_n)

bars = axes[0].barh(range(len(top_features_ebm)),
                   top_features_ebm['Importance'].values,
                   color=[COLOR_PALETTE['positive'] if x > 0 else COLOR_PALETTE['negative']
                          for x in top_features_ebm['Importance'].values],
                   alpha=0.7)
axes[0].set_yticks(range(len(top_features_ebm)))
axes[0].set_yticklabels(top_features_ebm['Feature'].values)
axes[0].set_xlabel('Feature Importance Score')
axes[0].set_title(f'Top {top_n} EBM Feature Importances\n',
                  fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')
axes[0].axvline(x=0, color='black', linewidth=0.5)

# Cumulative importance plot
cumulative_importance = np.cumsum(feature_importance_ebm['Abs_Importance'].values)
cumulative_importance = cumulative_importance / cumulative_importance[-1]

axes[1].plot(range(1, len(cumulative_importance) + 1),
            cumulative_importance * 100,
            'o-',
            color=COLOR_PALETTE['primary'],
            linewidth=2,
            markersize=4)

# Add markers for key points
for threshold in [0.5, 0.8, 0.9, 0.95]:
    idx = np.where(cumulative_importance >= threshold)[0]
    if len(idx) > 0:
        n_features = idx[0] + 1
        axes[1].axhline(y=threshold * 100, color='gray', linestyle='--', alpha=0.5)
        axes[1].plot(n_features, threshold * 100, 'ro', markersize=8)
        axes[1].annotate(f'{n_features} features\n({threshold*100:.0f}%)',
                        (n_features, threshold * 100),
                        xytext=(10, 10), textcoords='offset points',
                        fontsize=9, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

axes[1].set_xlabel('Number of Features')
axes[1].set_ylabel('Cumulative Importance (%)')
axes[1].set_title('Cumulative Feature Importance\n',
                  fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 105])

plt.suptitle('EBM Feature Importance Analysis\n',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 11.4 EBM PARTIAL DEPENDENCE & SHAP ANALYSIS
# --------------------------------------------------
print("\n📈 11.4 EBM PARTIAL DEPENDENCE & INTERACTION ANALYSIS")
print("-" * 50)

# Select top 6 features for detailed analysis
top_6_features = feature_importance_ebm.head(6)['Feature'].values

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for idx, (ax, feature) in enumerate(zip(axes.flat, top_6_features)):
    # Create bins for the feature
    unique_vals = X_train[feature].unique()

    if len(unique_vals) <= 20:
        # For categorical/low-cardinality features
        mean_response_by_value = []
        for val in sorted(unique_vals):
            mask = X_train[feature] == val
            if mask.sum() > 0:
                mean_response = y_train[mask].mean()
                mean_response_by_value.append((val, mean_response))

        if mean_response_by_value:
            vals, means = zip(*mean_response_by_value)
            bars = ax.bar(range(len(vals)), means,
                         color=COLOR_PALETTE['primary'], alpha=0.7)
            ax.set_xticks(range(len(vals)))
            ax.set_xticklabels([str(v) for v in vals], rotation=45)

    else:
        # For continuous features - create bins
        bins = np.percentile(X_train[feature], np.linspace(0, 100, 21))
        bin_centers = (bins[:-1] + bins[1:]) / 2
        bin_indices = np.digitize(X_train[feature], bins) - 1

        mean_response_by_bin = []
        for i in range(len(bins)-1):
            mask = bin_indices == i
            if mask.sum() > 0:
                mean_response = y_train[mask].mean()
                mean_response_by_bin.append((bin_centers[i], mean_response))

        if mean_response_by_bin:
            centers, means = zip(*mean_response_by_bin)
            ax.plot(centers, means, 'o-',
                   color=COLOR_PALETTE['primary'],
                   linewidth=2,
                   markersize=4)

    ax.set_xlabel(feature, fontsize=10)
    ax.set_ylabel('Mean Target Value', fontsize=10)
    ax.set_title(f'Partial Dependence: {feature}\n', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=y_train.mean(), color='red', linestyle='--', alpha=0.5,
               label=f'Overall Mean ({y_train.mean():.3f})')
    ax.legend(loc='best', fontsize=9)

plt.suptitle('Partial Dependence Plots for Top EBM Features\n',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 11.5 EBM INTERACTION HEATMAP & PAIRWISE ANALYSIS
# --------------------------------------------------
print("\n🔥 11.5 EBM INTERACTION ANALYSIS & HEATMAPS")
print("-" * 50)

# Calculate correlation matrix for top features
top_10_features = feature_importance_ebm.head(10)['Feature'].values
correlation_matrix_top = X_train[top_10_features].corr()

# Create interaction analysis
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Heatmap 1: Correlation matrix
sns.heatmap(correlation_matrix_top,
            cmap='RdBu_r',
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={"shrink": 0.8, "label": "Correlation"},
            annot=True,
            fmt='.2f',
            annot_kws={'size': 9},
            ax=axes[0])

axes[0].set_title('Correlation Matrix: Top 10 EBM Features\n',
                  fontsize=12, fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=9)
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0, fontsize=9)

# Create interaction strength visualization
interaction_matrix = np.zeros((len(top_10_features), len(top_10_features)))

# Calculate mutual information for each feature with target
mi_scores = mutual_info_classif(X_train[top_10_features], y_train, random_state=42)
mi_df = pd.DataFrame({'Feature': top_10_features, 'MI_Score': mi_scores})

# Create interaction matrix (simplified - could be enhanced with actual EBM interaction scores)
for i, feat1 in enumerate(top_10_features):
    for j, feat2 in enumerate(top_10_features):
        if i != j:
            # Simple interaction measure: product of individual MI scores * correlation
            corr = np.abs(correlation_matrix_top.iloc[i, j])
            interaction_matrix[i, j] = mi_df.iloc[i]['MI_Score'] * mi_df.iloc[j]['MI_Score'] * corr

# Normalize
if interaction_matrix.max() > 0:
    interaction_matrix = interaction_matrix / interaction_matrix.max()

# Mask diagonal
mask = np.eye(len(top_10_features), dtype=bool)
interaction_matrix_masked = np.ma.array(interaction_matrix, mask=mask)

# Heatmap 2: Interaction strength
im = axes[1].imshow(interaction_matrix_masked, cmap='YlOrRd', aspect='auto')
axes[1].set_xticks(range(len(top_10_features)))
axes[1].set_yticks(range(len(top_10_features)))
axes[1].set_xticklabels(top_10_features, rotation=45, ha='right', fontsize=9)
axes[1].set_yticklabels(top_10_features, fontsize=9)
axes[1].set_title('Feature Interaction Strength (Estimated)\n',
                  fontsize=12, fontweight='bold')

# Add colorbar
cbar = plt.colorbar(im, ax=axes[1], shrink=0.8)
cbar.set_label('Interaction Strength', fontsize=10)

# Annotate strongest interactions
strong_interactions = []
for i in range(len(top_10_features)):
    for j in range(i+1, len(top_10_features)):
        if interaction_matrix[i, j] > 0.5:  # Threshold for strong interaction
            strong_interactions.append((top_10_features[i], top_10_features[j], interaction_matrix[i, j]))

if strong_interactions:
    print("\n🔍 Strongest Feature Interactions (Estimated):")
    for feat1, feat2, strength in sorted(strong_interactions, key=lambda x: x[2], reverse=True)[:5]:
        print(f"  {feat1} ↔ {feat2}: {strength:.3f}")

plt.suptitle('EBM Feature Interaction Analysis\n',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 11.6 EBM PREDICTION DISTRIBUTION & CALIBRATION
# --------------------------------------------------
print("\n📊 11.6 EBM PREDICTION ANALYSIS & CALIBRATION")
print("-" * 50)

ebm_X_train = pd.Series(
    ebm_model.predict_proba(X_train)[:, 1],
    index=X_train.index,
    name="ebm_prob"
)

ebm_X_test = pd.Series(
    ebm_model.predict_proba(X_test)[:, 1],
    index=X_test.index,
    name="ebm_prob"
)


fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Distribution of predicted probabilities
axes[0, 0].hist(ebm_X_train, bins=50, alpha=0.7,
                color=COLOR_PALETTE['primary'], edgecolor='black', density=True)
axes[0, 0].axvline(x=0.5, color=COLOR_PALETTE['secondary'],
                   linestyle='--', linewidth=2, label='Decision Threshold 0.5')
axes[0, 0].set_xlabel('Predicted Probability (Class 1 = "yes")')
axes[0, 0].set_ylabel('Density')
axes[0, 0].set_title('EBM Training Predictions Distribution\n',
                     fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. ROC Curve
fpr_ebm, tpr_ebm, _ = roc_curve(y_test, y_pred_proba_ebm)
roc_auc_ebm = auc(fpr_ebm, tpr_ebm)

axes[0, 1].plot(fpr_ebm, tpr_ebm, 'b-',
                label=f'EBM ROC (AUC = {roc_auc_ebm:.3f})',
                linewidth=2)
axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Classifier')
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('EBM ROC Curve (Test Set)\n',
                     fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Calibration curve
from sklearn.calibration import calibration_curve

prob_true_ebm, prob_pred_ebm = calibration_curve(y_test, y_pred_proba_ebm, n_bins=10)

axes[1, 0].plot(prob_pred_ebm, prob_true_ebm, 's-',
                color=COLOR_PALETTE['primary'],
                linewidth=2,
                markersize=8,
                label='EBM Calibration')
axes[1, 0].plot([0, 1], [0, 1], 'k:',
                label='Perfect Calibration',
                linewidth=2)
axes[1, 0].fill_between(prob_pred_ebm,
                        prob_pred_ebm - 0.05,
                        prob_pred_ebm + 0.05,
                        alpha=0.2,
                        color=COLOR_PALETTE['neutral'],
                        label='±5% Margin')

axes[1, 0].set_xlabel('Mean Predicted Probability')
axes[1, 0].set_ylabel('Fraction of Positives')
axes[1, 0].set_title('EBM Calibration Curve\n',
                     fontsize=12, fontweight='bold')
axes[1, 0].legend(loc='upper left')
axes[1, 0].grid(True, alpha=0.3)

# 4. Prediction reliability diagram
pred_bins = np.linspace(0, 1, 11)
bin_centers = (pred_bins[:-1] + pred_bins[1:]) / 2
bin_indices = np.digitize(ebm_X_test, pred_bins) - 1

actual_rates = []
pred_rates = []
for i in range(len(pred_bins)-1):
    mask = bin_indices == i
    if mask.sum() > 0:
        actual_rates.append(y_test[mask].mean())
        pred_rates.append(ebm_X_test[mask].mean())
    else:
        actual_rates.append(np.nan)
        pred_rates.append(np.nan)

axes[1, 1].bar(bin_centers, actual_rates, width=0.08,
               color=COLOR_PALETTE['positive'], alpha=0.6,
               label='Actual Rate')
axes[1, 1].plot(bin_centers, pred_rates, 'o-',
                color=COLOR_PALETTE['primary'],
                linewidth=2,
                markersize=6,
                label='Predicted Rate')
axes[1, 1].set_xlabel('Predicted Probability Bin')
axes[1, 1].set_ylabel('Rate')
axes[1, 1].set_title('EBM Prediction Reliability Diagram\n',
                     fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xticks(bin_centers)
axes[1, 1].set_xticklabels([f'{c:.1f}' for c in bin_centers])

plt.suptitle('EBM Prediction Analysis & Calibration\n',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# --------------------------------------------------
# 11.7 EBM ENSEMBLE PREPARATION & MODEL PERSISTENCE (GLOBAL SPLIT SAFE)
# --------------------------------------------------
print("\n💾 11.7 EBM ENSEMBLE PREPARATION & MODEL PERSISTENCE")
print("-" * 50)

# ------------------------------------------------------------------
# Ensure predictions are INDEX-ALIGNED (CRITICAL FOR META)
# ------------------------------------------------------------------
ebm_X_train = pd.Series(
    ebm_model.predict_proba(X_train)[:, 1],
    index=X_train.index,
    name="ebm_prob"
)

ebm_X_test = pd.Series(
    ebm_model.predict_proba(X_test)[:, 1],
    index=X_test.index,
    name="ebm_prob"
)

# ------------------------------------------------------------------
# Create META-SAFE ensemble dictionary
# ------------------------------------------------------------------
ebm_ensemble_dict = {
    'model': ebm_model,
    'model_name': 'explainable_boosting_machine',
    'model_type': 'EBM',

    # Feature & training metadata
    'feature_names': X.columns.tolist(),
    'training_data_shape': X_train.shape,
    'training_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    # GLOBAL SPLIT CONTRACT
    'global_split': GLOBAL_SPLIT,

    # Cross-validation results (TRAIN ONLY)
    'cv_scores': cv_scores.tolist(),
    'cv_mean': float(cv_scores.mean()),
    'cv_std': float(cv_scores.std()),

    # Performance metrics (HOLDOUT)
    'performance_metrics': metrics_ebm,

    # Feature importance
    'feature_importance': feature_importance_ebm.to_dict(),
    'top_features': feature_importance_ebm.head(20)['Feature'].tolist(),

    # --------------------------------------------------
    # META-CRITICAL: INDEX-ALIGNED PREDICTIONS
    # --------------------------------------------------
    'train_predictions': {
        'index': ebm_X_train.index.tolist(),
        'probabilities': ebm_X_train.values.tolist()
    },
    'test_predictions': {
        'index': ebm_X_test.index.tolist(),
        'probabilities': ebm_X_test.values.tolist()
    },

    # Labels (index-aligned)
    'train_labels': y_train.loc[ebm_X_train.index].values.tolist(),
    'test_labels': y_test.loc[ebm_X_test.index].values.tolist(),

    # Model configuration
    'model_params': ebm_model.get_params(),

    # Dataset info
    'target_distribution': dict(y.value_counts()),
    'target_proportion': float(y.mean())
}

# ------------------------------------------------------------------
# SAVE ARTIFACTS
# ------------------------------------------------------------------
base_path = "./models"
os.makedirs(base_path, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
ebm_model_filename = f"{base_path}/ebm_model_{timestamp}.pkl"
ebm_ensemble_filename = f"{base_path}/ebm_ensemble_dict_{timestamp}.pkl"
ebm_joblib_filename = f"{base_path}/ebm_ensemble_joblib_{timestamp}.joblib"

with open(ebm_model_filename, 'wb') as f:
    pickle.dump(ebm_model, f)

with open(ebm_ensemble_filename, 'wb') as f:
    pickle.dump(ebm_ensemble_dict, f)

joblib.dump(ebm_ensemble_dict, ebm_joblib_filename)

print(f"✅ EBM model saved to: {ebm_model_filename}")
print(f"✅ EBM ensemble dict saved to: {ebm_ensemble_filename}")
print(f"✅ EBM ensemble dict saved (joblib) to: {ebm_joblib_filename}")

# ------------------------------------------------------------------
# OPTIONAL BUT STRONGLY RECOMMENDED: META TRAIN FRAME
# ------------------------------------------------------------------
ebm_meta_train = pd.DataFrame(
    {'ebm_prob': ebm_X_train},
    index=ebm_X_train.index
)

meta_train_path = f"{base_path}/ebm_meta_train_{timestamp}.csv"
ebm_meta_train.to_csv(meta_train_path)

print(f"📦 Meta-train frame saved: {meta_train_path}")

# ------------------------------------------------------------------
# FINAL VERIFICATION
# ------------------------------------------------------------------
print("\n🔍 Final Verification:")
print(f"Train preds: {len(ebm_X_train)} | Test preds: {len(ebm_X_test)}")
print(f"Index aligned: {ebm_X_train.index.equals(y_train.index)}")
print("✅ EBM READY FOR META-ENSEMBLE")


🌳 CELL 11: EXPLAINABLE BOOSTING MACHINE (EBM) - FULL ANALYSIS

💻 INSTALL REQUIRED PACKAGE:
!pip install interpret -q

If already installed, continue with the analysis below.


🔧 11.1 DATA PREPARATION (GLOBAL SPLIT)
--------------------------------------------------

✓ GLOBAL SPLIT APPLIED:
  Train: (36168, 29) | Pos rate: 0.117
  Test:  (9043, 29) | Pos rate: 0.117

✓ 10-fold CV configured on TRAIN only

🤖 11.2 EBM MODEL TRAINING & CROSS-VALIDATION
--------------------------------------------------
✓ EBM model initialized with configuration:
  • max_bins: 256
  • interactions: 10
  • learning_rate: 0.01
  • validation_size: 0.1

🔍 Performing 10-fold cross-validation...


KeyboardInterrupt: 

In [42]:
import pickletools

with open("/content/models/glass_brw_bundle_20251224_044027.joblib", "rb") as f:
    pickletools.dis(f)


    0: \x80 PROTO      4
    2: \x95 FRAME      1841
   11: }    EMPTY_DICT
   12: \x94 MEMOIZE    (as 0)
   13: (    MARK
   14: \x8c     SHORT_BINUNICODE 'model'
   21: \x94     MEMOIZE    (as 1)
   22: \x8c     SHORT_BINUNICODE '__main__'
   32: \x94     MEMOIZE    (as 2)
   33: \x8c     SHORT_BINUNICODE 'GLASS_BRW'
   44: \x94     MEMOIZE    (as 3)
   45: \x93     STACK_GLOBAL
   46: \x94     MEMOIZE    (as 4)
   47: )        EMPTY_TUPLE
   48: \x81     NEWOBJ
   49: \x94     MEMOIZE    (as 5)
   50: }        EMPTY_DICT
   51: \x94     MEMOIZE    (as 6)
   52: (        MARK
   53: \x8c         SHORT_BINUNICODE 'mode'
   59: \x94         MEMOIZE    (as 7)
   60: \x8c         SHORT_BINUNICODE 'strict'
   68: \x94         MEMOIZE    (as 8)
   69: \x8c         SHORT_BINUNICODE 'min_precision_subscribe'
   94: \x94         MEMOIZE    (as 9)
   95: G            BINFLOAT   0.1
  104: \x8c         SHORT_BINUNICODE 'min_precision_not_subscribe'
  133: \x94         MEMOIZE    (as 10)
  134: 

ValueError: stack not empty after STOP: [any, mark, any, any, any, mark, any, any, any, float, any, float, any, float, any, float, any, int, any, int, any, int, any, int, any, int, any, int, any, any, any, any, any, mark, str, int, str, int, any, int, any, int, any, int, any, float, any, float, str, float, str, float, str, float, str, float, any, float, any, float, any, float, any, float, str, str, any, int, any, float, any, float, any, float, any, any, any, any, any, mark, any, any, any, int, any, any, any, bool, any, bool, any, int, str, int, any, int, any, bool, str, any, any, None, str, str, str, int, str, int, str, int, str, float, str, any, str, None, str, float, str, None, str, float, any, any]

In [155]:
# ============================================================
# CELL: META-EBM (STAGE 4) — FINAL ASSEMBLY
# ============================================================

print("\n" + "="*80)
print("🎯 META-EBM (STAGE 4) — FINAL CASCADE ASSEMBLY")
print("="*80)

import pickle
import joblib
import numpy as np
import pandas as pd
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from datetime import datetime

# --------------------------------------------------
# 1. LOAD STAGE ARTIFACTS (NO RE-COMPUTATION)
# --------------------------------------------------
print("\n📂 Loading stage ensemble artifacts...")

lr_dict = joblib.load("/content/models/lr_ensemble_joblib_20251224_042425.joblib")
ebm_dict = joblib.load("/content/models/ebm_ensemble_joblib_20251224_060127.joblib")
glass_bundle = joblib.load("/content/models/glass_brw_bundle_safe_20251224_083820.joblib")

print("✅ Stage 1–3 artifacts loaded")

# --------------------------------------------------
# 2. VERIFY INDEX ALIGNMENT
# --------------------------------------------------
print("\n🔍 Verifying index alignment...")

lr_train_len = len(lr_dict["train_predictions"])
lr_test_len = len(lr_dict["test_predictions"])
glass_train_len = len(glass_bundle["train_idx"])
glass_test_len = len(glass_bundle["test_idx"])
ebm_train_len = len(ebm_dict["train_predictions"]["index"])
ebm_test_len = len(ebm_dict["test_predictions"]["index"])

print(f"   LR:    Train={lr_train_len:,}, Test={lr_test_len:,}")
print(f"   GLASS: Train={glass_train_len:,}, Test={glass_test_len:,}")
print(f"   EBM:   Train={ebm_train_len:,}, Test={ebm_test_len:,}")

assert lr_train_len == glass_train_len == ebm_train_len, "Train set size mismatch!"
assert lr_test_len == glass_test_len == ebm_test_len, "Test set size mismatch!"
print("✅ All models use consistent train/test splits")

# --------------------------------------------------
# 3. FIX GLASS-BRW COLUMN SWAP
# --------------------------------------------------
print("\n🔧 Fixing Glass-BRW column swap...")

# 🔥 DEBUG: Check current format
print(f"train_proba shape: {glass_bundle['train_proba'].shape}")
print(f"First row: {glass_bundle['train_proba'][0]}")
print(f"Second row: {glass_bundle['train_proba'][1]}")

# Swap columns: [P(NOT_SUB), P(SUB)] → [P(SUB), P(NOT_SUB)]
glass_bundle["train_proba"] = glass_bundle["train_proba"][:, [1, 0]]
glass_bundle["test_proba"] = glass_bundle["test_proba"][:, [1, 0]]
print("✅ Columns swapped: Column 0 = P(SUB), Column 1 = P(NOT_SUB)")

# --------------------------------------------------
# 4. EXTRACT INDEX-ALIGNED PREDICTIONS
# --------------------------------------------------
print("\n🔧 Extracting aligned predictions...")

train_idx = glass_bundle["train_idx"]
test_idx = glass_bundle["test_idx"]

# LR predictions
lr_train = pd.Series(lr_dict["train_predictions"], index=train_idx, name="lr_prob")
lr_test = pd.Series(lr_dict["test_predictions"], index=test_idx, name="lr_prob")

# EBM predictions
ebm_train = pd.Series(
    ebm_dict["train_predictions"]["probabilities"],
    index=ebm_dict["train_predictions"]["index"],
    name="ebm_prob"
)
ebm_test = pd.Series(
    ebm_dict["test_predictions"]["probabilities"],
    index=ebm_dict["test_predictions"]["index"],
    name="ebm_prob"
)

# Glass predictions (after swap)
glass_train = pd.Series(
    glass_bundle["train_proba"][:, 0],
    index=train_idx,
    name="glass_prob"
)
glass_test = pd.Series(
    glass_bundle["test_proba"][:, 0],
    index=test_idx,
    name="glass_prob"
)

# Coverage flags
glass_covered_train = pd.Series(
    glass_bundle["train_covered"],
    index=train_idx,
    name="glass_covered"
)
glass_covered_test = pd.Series(
    glass_bundle["test_covered"],
    index=test_idx,
    name="glass_covered"
)

# Labels
y_train = pd.Series(lr_dict["train_labels"], index=train_idx)
y_test = pd.Series(lr_dict["test_labels"], index=test_idx)

print(f"✅ Predictions extracted:")
print(f"   Train: {len(lr_train):,} samples")
print(f"   Test:  {len(lr_test):,} samples")

print(f"\n📊 Glass probability sanity check:")
print(f"   Train mean P(SUB): {glass_train.mean():.4f}")
print(f"   Test mean P(SUB):  {glass_test.mean():.4f}")
print(f"   Train coverage:    {glass_covered_train.mean():.1%}")
print(f"   Test coverage:     {glass_covered_test.mean():.1%}")

# --------------------------------------------------
# 5. BUILD META FEATURE GEOMETRY
# --------------------------------------------------
print("\n🧠 Building meta feature geometry...")

# Combine base features
meta_train = pd.concat([lr_train, glass_train, ebm_train, glass_covered_train], axis=1)
meta_test = pd.concat([lr_test, glass_test, ebm_test, glass_covered_test], axis=1)

# --- DISAGREEMENT FEATURES ---
meta_train["lr_glass_diff"] = (meta_train.lr_prob - meta_train.glass_prob).abs()
meta_train["lr_ebm_diff"] = (meta_train.lr_prob - meta_train.ebm_prob).abs()
meta_train["glass_ebm_diff"] = (meta_train.glass_prob - meta_train.ebm_prob).abs()

meta_test["lr_glass_diff"] = (meta_test.lr_prob - meta_test.glass_prob).abs()
meta_test["lr_ebm_diff"] = (meta_test.lr_prob - meta_test.ebm_prob).abs()
meta_test["glass_ebm_diff"] = (meta_test.glass_prob - meta_test.ebm_prob).abs()

# --- CONFIDENCE FEATURES ---
P_tr = meta_train[["lr_prob", "glass_prob", "ebm_prob"]].values
P_te = meta_test[["lr_prob", "glass_prob", "ebm_prob"]].values

meta_train["max_confidence"] = P_tr.max(axis=1)
meta_train["min_confidence"] = P_tr.min(axis=1)
meta_train["std_confidence"] = P_tr.std(axis=1)
meta_train["mean_confidence"] = P_tr.mean(axis=1)

meta_test["max_confidence"] = P_te.max(axis=1)
meta_test["min_confidence"] = P_te.min(axis=1)
meta_test["std_confidence"] = P_te.std(axis=1)
meta_test["mean_confidence"] = P_te.mean(axis=1)

# --- VOTING FEATURES ---
meta_train["unanimous_sub"] = ((meta_train.lr_prob >= 0.5) &
                               (meta_train.glass_prob >= 0.5) &
                               (meta_train.ebm_prob >= 0.5)).astype(int)
meta_train["unanimous_not_sub"] = ((meta_train.lr_prob < 0.5) &
                                   (meta_train.glass_prob < 0.5) &
                                   (meta_train.ebm_prob < 0.5)).astype(int)
meta_train["majority_vote"] = ((meta_train.lr_prob >= 0.5).astype(int) +
                               (meta_train.glass_prob >= 0.5).astype(int) +
                               (meta_train.ebm_prob >= 0.5).astype(int))

meta_test["unanimous_sub"] = ((meta_test.lr_prob >= 0.5) &
                              (meta_test.glass_prob >= 0.5) &
                              (meta_test.ebm_prob >= 0.5)).astype(int)
meta_test["unanimous_not_sub"] = ((meta_test.lr_prob < 0.5) &
                                  (meta_test.glass_prob < 0.5) &
                                  (meta_test.ebm_prob < 0.5)).astype(int)
meta_test["majority_vote"] = ((meta_test.lr_prob >= 0.5).astype(int) +
                              (meta_test.glass_prob >= 0.5).astype(int) +
                              (meta_test.ebm_prob >= 0.5).astype(int))

# --- AGREEMENT LEVELS ---
meta_train["agreement_level"] = 0
meta_train.loc[(meta_train["lr_glass_diff"] < 0.2) &
               (meta_train["lr_ebm_diff"] < 0.2), "agreement_level"] = 1  # All agree
meta_train.loc[(meta_train["majority_vote"] == 2), "agreement_level"] = 2  # Majority agree
meta_train.loc[(meta_train["majority_vote"] == 1) | (meta_train["majority_vote"] == 0), "agreement_level"] = 3  # Split

meta_test["agreement_level"] = 0
meta_test.loc[(meta_test["lr_glass_diff"] < 0.2) &
              (meta_test["lr_ebm_diff"] < 0.2), "agreement_level"] = 1
meta_test.loc[(meta_test["majority_vote"] == 2), "agreement_level"] = 2
meta_test.loc[(meta_test["majority_vote"] == 1) | (meta_test["majority_vote"] == 0), "agreement_level"] = 3

# --- CONFIDENCE CATEGORIES ---
def assign_confidence_category(row):
    if row["max_confidence"] > 0.8:
        return 2  # high
    elif row["max_confidence"] < 0.3:
        return 0  # low
    elif row["std_confidence"] > 0.3:
        return 3  # conflicting
    else:
        return 1  # medium

meta_train["confidence_category"] = meta_train.apply(assign_confidence_category, axis=1)
meta_test["confidence_category"] = meta_test.apply(assign_confidence_category, axis=1)

# --- DOMINANT MODEL ---
meta_train["dominant_model"] = meta_train[["lr_prob", "glass_prob", "ebm_prob"]].idxmax(axis=1)
meta_test["dominant_model"] = meta_test[["lr_prob", "glass_prob", "ebm_prob"]].idxmax(axis=1)

model_map = {"lr_prob": 0, "glass_prob": 1, "ebm_prob": 2}
meta_train["dominant_model_code"] = meta_train["dominant_model"].map(model_map)
meta_test["dominant_model_code"] = meta_test["dominant_model"].map(model_map)

# --- PROBABILITY BUCKETS ---
for model in ["lr_prob", "glass_prob", "ebm_prob"]:
    meta_train[f"{model}_bucket"] = (meta_train[model] * 10).round() / 10
    meta_test[f"{model}_bucket"] = (meta_test[model] * 10).round() / 10

# --- DISTANCE TO BOUNDARY ---
meta_train["lr_distance_to_boundary"] = abs(meta_train["lr_prob"] - 0.5)
meta_train["glass_distance_to_boundary"] = abs(meta_train["glass_prob"] - 0.5)
meta_train["ebm_distance_to_boundary"] = abs(meta_train["ebm_prob"] - 0.5)

meta_test["lr_distance_to_boundary"] = abs(meta_test["lr_prob"] - 0.5)
meta_test["glass_distance_to_boundary"] = abs(meta_test["glass_prob"] - 0.5)
meta_test["ebm_distance_to_boundary"] = abs(meta_test["ebm_prob"] - 0.5)

# --- CLEAN UP ---
meta_train = meta_train.drop(["dominant_model"], axis=1)
meta_test = meta_test.drop(["dominant_model"], axis=1)

print(f"✅ Meta feature space ready: {meta_train.shape}")
print(f"   Features: {list(meta_train.columns)}")

# --------------------------------------------------
# 6. TRAIN META-EBM ROUTER
# --------------------------------------------------
print("\n🎯 Training META-EBM router...")

meta_ebm = ExplainableBoostingClassifier(
    max_bins=16,
    interactions=5,
    learning_rate=0.01,
    min_samples_leaf=10,
    max_leaves=3,
    random_state=42
)

meta_ebm.fit(meta_train, y_train)
print("✅ Meta-EBM trained")

# --------------------------------------------------
# 7. EVALUATION
# --------------------------------------------------
print("\n📊 Evaluating META-EBM on test set...")

meta_prob = meta_ebm.predict_proba(meta_test)[:, 1]
meta_pred = (meta_prob >= 0.5).astype(int)

metrics = {
    "accuracy": accuracy_score(y_test, meta_pred),
    "precision": precision_score(y_test, meta_pred, zero_division=0),
    "recall": recall_score(y_test, meta_pred, zero_division=0),
    "f1": f1_score(y_test, meta_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, meta_prob)
}

print("\n" + "="*80)
print("📊 META-EBM PERFORMANCE (TEST SET)")
print("="*80)
for k, v in metrics.items():
    print(f"   {k:10s}: {v:.4f}")

cm = confusion_matrix(y_test, meta_pred)
print(f"\n📊 Confusion Matrix:")
print(f"                Pred NOT_SUB  Pred SUBSCRIBE")
print(f"Actual NOT_SUB      {cm[0,0]:5,}          {cm[0,1]:5,}")
print(f"Actual SUBSCRIBE    {cm[1,0]:5,}          {cm[1,1]:5,}")

# Compare to base models
print("\n" + "="*80)
print("📊 COMPARISON: META-EBM vs BASE MODELS (TEST SET)")
print("="*80)

# LR baseline
lr_pred_test = (lr_test >= 0.5).astype(int)
lr_metrics = {
    "precision": precision_score(y_test, lr_pred_test, zero_division=0),
    "recall": recall_score(y_test, lr_pred_test, zero_division=0),
    "f1": f1_score(y_test, lr_pred_test, zero_division=0)
}

# EBM baseline
ebm_pred_test = (ebm_test >= 0.5).astype(int)
ebm_metrics = {
    "precision": precision_score(y_test, ebm_pred_test, zero_division=0),
    "recall": recall_score(y_test, ebm_pred_test, zero_division=0),
    "f1": f1_score(y_test, ebm_pred_test, zero_division=0)
}

# GLASS baseline
glass_covered_mask = glass_covered_test.values
if glass_covered_mask.sum() > 0:
    glass_pred_test = (glass_test >= 0.5).astype(int)
    glass_metrics = {
        "precision": precision_score(y_test[glass_covered_mask], glass_pred_test[glass_covered_mask], zero_division=0),
        "recall": recall_score(y_test[glass_covered_mask], glass_pred_test[glass_covered_mask], zero_division=0),
        "coverage": glass_covered_mask.mean()
    }
else:
    glass_metrics = {"precision": 0.0, "recall": 0.0, "coverage": 0.0}

print(f"\n   LR:       Prec={lr_metrics['precision']:.3f}, Rec={lr_metrics['recall']:.3f}, F1={lr_metrics['f1']:.3f}")
print(f"   EBM:      Prec={ebm_metrics['precision']:.3f}, Rec={ebm_metrics['recall']:.3f}, F1={ebm_metrics['f1']:.3f}")
print(f"   GLASS:    Prec={glass_metrics['precision']:.3f}, Rec={glass_metrics['recall']:.3f}, Cov={glass_metrics['coverage']:.1%}")
print(f"   META-EBM: Prec={metrics['precision']:.3f}, Rec={metrics['recall']:.3f}, F1={metrics['f1']:.3f}")

# --------------------------------------------------
# 8. SAVE META CHECKPOINT
# --------------------------------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
checkpoint = {
    "meta_model": meta_ebm,
    "metrics": metrics,
    "comparison": {
        "lr": lr_metrics,
        "ebm": ebm_metrics,
        "glass": glass_metrics,
        "meta": metrics
    },
    "meta_features": meta_train.columns.tolist(),
    "training_date": timestamp,
    "train_idx": train_idx,
    "test_idx": test_idx
}

fname = f"/content/models/meta_ebm_checkpoint_{timestamp}.pkl"
with open(fname, "wb") as f:
    pickle.dump(checkpoint, f)

print(f"\n💾 Saved meta checkpoint → {fname}")
print("\n" + "="*80)
print("🎉 4-STAGE GLASS-BOX CASCADE COMPLETE")
print("="*80)


🎯 META-EBM (STAGE 4) — FINAL CASCADE ASSEMBLY

📂 Loading stage ensemble artifacts...
✅ Stage 1–3 artifacts loaded

🔍 Verifying index alignment...
   LR:    Train=36,168, Test=9,043
   GLASS: Train=36,168, Test=9,043
   EBM:   Train=36,168, Test=9,043
✅ All models use consistent train/test splits

🔧 Fixing Glass-BRW column swap...
train_proba shape: (36168, 2)
First row: [0.92186566 0.07813434]
Second row: [0.88 0.12]
✅ Columns swapped: Column 0 = P(SUB), Column 1 = P(NOT_SUB)

🔧 Extracting aligned predictions...
✅ Predictions extracted:
   Train: 36,168 samples
   Test:  9,043 samples

📊 Glass probability sanity check:
   Train mean P(SUB): 0.0965
   Test mean P(SUB):  0.0966
   Train coverage:    96.6%
   Test coverage:     96.4%

🧠 Building meta feature geometry...
✅ Meta feature space ready: (36168, 23)
   Features: ['lr_prob', 'glass_prob', 'ebm_prob', 'glass_covered', 'lr_glass_diff', 'lr_ebm_diff', 'glass_ebm_diff', 'max_confidence', 'min_confidence', 'std_confidence', 'mean_con